In [12]:
import pandas as pd

# 1. ملف الأحداث
events = pd.read_csv("Sample_Game_1_RawEventsData.csv")
print("--- Events Head ---")
print(events.head())

# 2. ملف تتبع الفريق الأول (تخطي أول سطرين بتوع الهيدر المعقد)
home = pd.read_csv("Sample_Game_1_RawTrackingData_Home_Team.csv", skiprows=2)
print("\n--- Home Tracking Head ---")
print(home.head())

# 3. ملف تتبع الفريق الثاني
away = pd.read_csv("Sample_Game_1_RawTrackingData_Away_Team.csv", skiprows=2)
print("\n--- Away Tracking Head ---")
print(away.head())

--- Events Head ---
   Team       Type   Subtype  Period  Start Frame  Start Time [s]  End Frame  \
0  Away  SET PIECE  KICK OFF       1            1            0.04          0   
1  Away       PASS       NaN       1            1            0.04          3   
2  Away       PASS       NaN       1            3            0.12         17   
3  Away       PASS       NaN       1           45            1.80         61   
4  Away       PASS       NaN       1           77            3.08         96   

   End Time [s]      From        To  Start X  Start Y  End X  End Y  
0          0.00  Player19       NaN      NaN      NaN    NaN    NaN  
1          0.12  Player19  Player21     0.45     0.39   0.55   0.43  
2          0.68  Player21  Player15     0.55     0.43   0.58   0.21  
3          2.44  Player15  Player19     0.55     0.19   0.45   0.31  
4          3.84  Player19  Player21     0.45     0.32   0.49   0.47  

--- Home Tracking Head ---
   Period  Frame  Time [s]  Player11  Unnamed: 4  P

In [13]:
# ==============================================================================
# Step 1: Full Match Ingestion, Column Alignment, and Metric Conversion
# Dataset: Metrica Sports Sample Game 1 (Full 90 Minutes)
# ==============================================================================

import pandas as pd
import numpy as np

# Standard Pitch Dimensions (Meters)
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

# Raw File Paths
HOME_FILE = "Sample_Game_1_RawTrackingData_Home_Team.csv"
AWAY_FILE = "Sample_Game_1_RawTrackingData_Away_Team.csv"
EVENTS_FILE = "Sample_Game_1_RawEventsData.csv"


def clean_tracking_columns(df, prefix="Home"):
    """
    Rename raw Metrica columns to structured entity pairs:
    [prefix_PlayerName_x, prefix_PlayerName_y] and [Ball_x, Ball_y].
    """
    cols = list(df.columns)
    new_cols = ["Period", "Frame", "Time [s]"]
    
    # Iterate through player tracking pairs (columns index 3 to n-2)
    i = 3
    while i < len(cols) - 2:
        player_name = cols[i]
        new_cols.append(f"{prefix}_{player_name}_x")
        new_cols.append(f"{prefix}_{player_name}_y")
        i += 2
        
    # Final two columns represent Ball X and Ball Y
    new_cols.append("Ball_x")
    new_cols.append("Ball_y")
    df.columns = new_cols
    return df


def to_metric_coordinates(df):
    """
    Scale normalized 0.0-1.0 Metrica coordinates to metric pitch bounds.
    Y-coordinates are flipped to align with international football standards.
    """
    x_cols = [c for c in df.columns if c.endswith("_x")]
    y_cols = [c for c in df.columns if c.endswith("_y")]
    
    for c in x_cols:
        df[c] = df[c] * PITCH_LENGTH
        
    for c in y_cols:
        df[c] = (1.0 - df[c]) * PITCH_WIDTH
        
    return df


print("Executing Step 1: Processing full match tracking and events...")

# 1. Ingest full tracking datasets (skipping multi-index headers)
df_home_raw = pd.read_csv(HOME_FILE, skiprows=2)
df_away_raw = pd.read_csv(AWAY_FILE, skiprows=2)

# 2. Standardize column names
df_home = clean_tracking_columns(df_home_raw, prefix="Home")
df_away = clean_tracking_columns(df_away_raw, prefix="Away")

# 3. Transform to standard metric coordinates
df_home = to_metric_coordinates(df_home)
df_away = to_metric_coordinates(df_away)

# 4. Ingest and scale events dataset
df_events = pd.read_csv(EVENTS_FILE)

for col in ["Start X", "End X"]:
    if col in df_events.columns:
        df_events[col] = df_events[col] * PITCH_LENGTH

for col in ["Start Y", "End Y"]:
    if col in df_events.columns:
        df_events[col] = (1.0 - df_events[col]) * PITCH_WIDTH

# 5. Output Verification
print("Step 1 successfully completed for the full match:")
print(f"- Total Home Frames: {len(df_home):,}")
print(f"- Total Away Frames: {len(df_away):,}")
print(f"- Total Match Events: {len(df_events):,}")
print(f"- Total Match Duration: {df_home['Time [s]'].max() / 60:.2f} minutes")

Executing Step 1: Processing full match tracking and events...
Step 1 successfully completed for the full match:
- Total Home Frames: 145,006
- Total Away Frames: 145,006
- Total Match Events: 1,745
- Total Match Duration: 96.67 minutes


In [14]:
# ==============================================================================
# Step 2: Temporal Synchronization, Unified Tracking Merge & Window Slicing
# ==============================================================================

import pandas as pd
import numpy as np

# 1. Identify Away player columns strictly (excluding metadata and ball)
metadata_and_ball = ["Period", "Frame", "Time [s]", "Ball_x", "Ball_y"]
away_player_cols = [c for c in df_away.columns if c not in metadata_and_ball]

# 2. Merge Home and Away tracking data on Frame and Period
df_tracking = pd.merge(
    df_home,
    df_away[["Frame"] + away_player_cols],
    on="Frame",
    how="inner"
)

# 3. Organize column structure: Metadata -> Ball -> All Players
meta_cols = ["Period", "Frame", "Time [s]", "Ball_x", "Ball_y"]
player_cols = [c for c in df_tracking.columns if c not in meta_cols]
df_tracking = df_tracking[meta_cols + player_cols]


# 4. Slicing function for target match window
def get_match_window(tracking_df, events_df, start_minute=0.0, end_minute=5.0):
    """
    Extract synchronized tracking frames and match events for a specific minute interval.
    """
    start_sec = start_minute * 60.0
    end_sec = end_minute * 60.0
    
    # Slice tracking by elapsed time in seconds
    window_tracking = tracking_df[
        (tracking_df["Time [s]"] >= start_sec) & 
        (tracking_df["Time [s]"] <= end_sec)
    ].copy().reset_index(drop=True)
    
    if window_tracking.empty:
        return window_tracking, pd.DataFrame()
    
    start_frame = window_tracking["Frame"].min()
    end_frame = window_tracking["Frame"].max()
    period = window_tracking["Period"].iloc[0]
    
    # Synchronize events within the exact frame window and match period
    window_events = events_df[
        (events_df["Period"] == period) &
        (events_df["Start Frame"] >= start_frame) &
        (events_df["Start Frame"] <= end_frame)
    ].copy().reset_index(drop=True)
    
    return window_tracking, window_events


# 5. Extract the first 5 minutes (0:00 to 5:00)
test_tracking_5m, test_events_5m = get_match_window(
    df_tracking, 
    df_events, 
    start_minute=0.0, 
    end_minute=5.0
)

# 6. Verification Outputs
print("Step 2 completed successfully:")
print(f"- Unified Tracking DataFrame shape: {df_tracking.shape}")
print(f"- Total tracked player coordinate columns: {len(player_cols)}")
print(f"\n--- Evaluation Window (0 to 5 Minutes) ---")
print(f"- Window Tracking Frames: {len(test_tracking_5m):,}")
print(f"- Window Events Count: {len(test_events_5m)}")
print(f"- Start Frame: {test_tracking_5m['Frame'].min()} | End Frame: {test_tracking_5m['Frame'].max()}")
print("\nFirst 3 Synchronized Events in Window:")
print(test_events_5m[["Type", "Subtype", "Team", "From", "To", "Start Frame", "Start Time [s]"]].head(3))

Step 2 completed successfully:
- Unified Tracking DataFrame shape: (145006, 61)
- Total tracked player coordinate columns: 56

--- Evaluation Window (0 to 5 Minutes) ---
- Window Tracking Frames: 7,500
- Window Events Count: 103
- Start Frame: 1 | End Frame: 7500

First 3 Synchronized Events in Window:
        Type   Subtype  Team      From        To  Start Frame  Start Time [s]
0  SET PIECE  KICK OFF  Away  Player19       NaN            1            0.04
1       PASS       NaN  Away  Player19  Player21            1            0.04
2       PASS       NaN  Away  Player21  Player15            3            0.12


In [15]:
# تنظيف أسماء الأعمدة من أي مسافات زائدة
df_events.columns = df_events.columns.str.strip()

# تطبيق التحويل المتري بشكل آمن مرة واحدة فقط
if df_events["Start X"].dropna().max() <= 2.0:
    for col in ["Start X", "End X"]:
        if col in df_events.columns:
            df_events[col] = df_events[col] * 105.0

    for col in ["Start Y", "End Y"]:
        if col in df_events.columns:
            df_events[col] = (1.0 - df_events[col]) * 68.0
    print("✅ تم تحويل الإحداثيات إلى أمتار بنجاح (105m × 68m).")
else:
    print("⚠️ الإحداثيات بالأمتار بالفعل، تم تجاوز التحويل لمنع التكرار.")

# فحص الإحداثيات بعد التحويل المظبوط
print("\n--- فحص الإحداثيات بعد التحويل المتري الصحيح ---")
print(df_events[["Type", "Start X", "Start Y", "End X", "End Y"]].dropna().head(3))
print("Max X:", df_events["Start X"].max())
print("Max Y:", df_events["Start Y"].max())

⚠️ الإحداثيات بالأمتار بالفعل، تم تجاوز التحويل لمنع التكرار.

--- فحص الإحداثيات بعد التحويل المتري الصحيح ---
   Type  Start X  Start Y  End X  End Y
1  PASS    47.25    41.48  57.75  38.76
2  PASS    57.75    38.76  60.90  53.72
3  PASS    57.75    55.08  47.25  46.92
Max X: 106.05
Max Y: 69.36


In [16]:
print("--- فحص الإحداثيات بعد Step 1 مباشرة ---")
print(df_events[["Type", "Start X", "Start Y", "End X", "End Y"]].dropna().head(3))
print("Max X:", df_events["Start X"].max())
print("Max Y:", df_events["Start Y"].max())

--- فحص الإحداثيات بعد Step 1 مباشرة ---
   Type  Start X  Start Y  End X  End Y
1  PASS    47.25    41.48  57.75  38.76
2  PASS    57.75    38.76  60.90  53.72
3  PASS    57.75    55.08  47.25  46.92
Max X: 106.05
Max Y: 69.36


In [165]:
import pandas as pd



# 2. فلترة كل التسديدات والأهداف
shots = df_events[df_events["Type"] == "SHOT"].copy()

# إضافة عمود للدقائق المحسوبة من الثواني لتسهيل القراءة
shots["Minute_Calculated"] = (shots["Start Time [s]"] / 60.0).round(2)

# تصفية الأهداف فقط
goals = shots[shots["Subtype"].str.contains("GOAL", na=False)]

print("=" * 60)
print("🎯 قائمة جميع الأهداف المسجلة في المباراة:")
print("=" * 60)

if not goals.empty:
    display_cols = [
        "Period", "Start Time [s]", "Minute_Calculated", 
        "Team", "From", "Type", "Subtype", "Start Frame"
    ]
    print(goals[display_cols].to_string(index=False))
else:
    print("لم يتم العثور على أهداف موسومة بـ GOAL مباشرة.")

print("\n" + "=" * 60)
print("🔍 جميع التسديدات في المباراة (Shots) للتأكد من تفاصيلها:")
print("=" * 60)
print(shots[["Period", "Minute_Calculated", "Team", "From", "Subtype", "Start Frame"]].to_string(index=False))

🎯 قائمة جميع الأهداف المسجلة في المباراة:
 Period  Start Time [s]  Minute_Calculated Team     From Type             Subtype  Start Frame
      1           91.56               1.53 Home  Player9 SHOT HEAD-ON TARGET-GOAL         2289
      2         3600.20              60.00 Home Player10 SHOT      ON TARGET-GOAL        90005
      2         3961.28              66.02 Home  Player9 SHOT      ON TARGET-GOAL        99032

🔍 جميع التسديدات في المباراة (Shots) للتأكد من تفاصيلها:
 Period  Minute_Calculated Team     From              Subtype  Start Frame
      1               1.53 Home  Player9  HEAD-ON TARGET-GOAL         2289
      1               3.95 Home Player10       OFF TARGET-OUT         5923
      1               5.17 Away Player21       OFF TARGET-OUT         7753
      1               6.42 Home  Player8      ON TARGET-SAVED         9628
      1              12.18 Home  Player8  HEAD-OFF TARGET-OUT        18270
      1              12.98 Away Player24      ON TARGET-SAVED        1

In [166]:
# Check raw vs scaled coordinates in df_events and df_tracking
print("--- Check Events Coordinates ---")
print(df_events[["Type", "Start X", "Start Y", "End X", "End Y"]].dropna().head(3))

print("\n--- Check Tracking Coordinates (Home_Player11 and Ball) ---")
print(df_tracking[["Frame", "Home_Player11_x", "Home_Player11_y", "Ball_x", "Ball_y"]].head(3))

--- Check Events Coordinates ---
   Type  Start X  Start Y  End X  End Y
1  PASS    47.25    41.48  57.75  38.76
2  PASS    57.75    38.76  60.90  53.72
3  PASS    57.75    55.08  47.25  46.92

--- Check Tracking Coordinates (Home_Player11 and Ball) ---
   Frame  Home_Player11_x  Home_Player11_y    Ball_x    Ball_y
0      1           0.0861         35.19816  47.74560  41.67788
1      2           0.1008         35.19816  52.12725  40.35392
2      3           0.1197         35.19816  56.40180  39.06192


In [167]:
# ==============================================================================
# Comprehensive Data Integrity & Sanity Check (Full Match Audit)
# ==============================================================================

import numpy as np
import pandas as pd

PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

print("=" * 70)
print("🔍 1. SPATIAL BOUNDS CHECK (Checking for proper metric conversion)")
print("=" * 70)

# Check df_events bounds
ev_x_max = max(df_events["Start X"].dropna().max(), df_events["End X"].dropna().max())
ev_x_min = min(df_events["Start X"].dropna().min(), df_events["End X"].dropna().min())
ev_y_max = max(df_events["Start Y"].dropna().max(), df_events["End Y"].dropna().max())
ev_y_min = min(df_events["Start Y"].dropna().min(), df_events["End Y"].dropna().min())

print(f"• Events X Bounds : [{ev_x_min:.2f}m  to  {ev_x_max:.2f}m]  (Target: 0 to 105m)")
print(f"• Events Y Bounds : [{ev_y_min:.2f}m  to  {ev_y_max:.2f}m]  (Target: 0 to 68m)")

# Check df_tracking bounds
tr_x_cols = [c for c in df_tracking.columns if c.endswith("_x")]
tr_y_cols = [c for c in df_tracking.columns if c.endswith("_y")]

tr_x_max = df_tracking[tr_x_cols].max().max()
tr_x_min = df_tracking[tr_x_cols].min().min()
tr_y_max = df_tracking[tr_y_cols].max().max()
tr_y_min = df_tracking[tr_y_cols].min().min()

print(f"• Tracking X Bounds: [{tr_x_min:.2f}m  to  {tr_x_max:.2f}m]  (Target: ~0 to 105m)")
print(f"• Tracking Y Bounds: [{tr_y_min:.2f}m  to  {tr_y_max:.2f}m]  (Target: ~0 to 68m)")

if ev_x_max > 1.0 and tr_x_max > 1.0:
    print("✅ STATUS: Both Events and Tracking are 100% in METRIC COORDINATES.")
else:
    print("❌ WARNING: One of the datasets is still in normalized percentage format!")


print("\n" + "=" * 70)
print("⏱️ 2. TEMPORAL & SYNCHRONIZATION CHECK")
print("=" * 70)
min_tr_frame = df_tracking["Frame"].min()
max_tr_frame = df_tracking["Frame"].max()
min_ev_frame = df_events["Start Frame"].dropna().min()
max_ev_frame = df_events["End Frame"].dropna().max()

print(f"• Tracking Frame Range: {min_tr_frame:,} ➔ {max_tr_frame:,} ({len(df_tracking):,} frames)")
print(f"• Events Frame Range  : {int(min_ev_frame):,} ➔ {int(max_ev_frame):,}")

# Check if any event falls outside tracking frames
events_outside = df_events[(df_events["Start Frame"] < min_tr_frame) | (df_events["Start Frame"] > max_tr_frame)]
print(f"• Events outside tracking bounds: {len(events_outside)}")
if len(events_outside) == 0:
    print("✅ STATUS: Perfect 1-to-1 temporal frame synchronization.")


print("\n" + "=" * 70)
print("🧭 3. GOALKEEPER & FIELD ORIENTATION AUDIT (Attacking Direction Ground-Truth)")
print("=" * 70)

# Check Home Goalkeeper (Player11) and Away Goalkeeper (Player25) across Period 1 and Period 2
for p in [1, 2]:
    p_df = df_tracking[df_tracking["Period"] == p]
    if not p_df.empty:
        h_gk_x = p_df["Home_Player11_x"].dropna().mean()
        a_gk_x = p_df.get("Away_Player25_x", p_df.get("Away_Player15_x", pd.Series())).dropna().mean()
        
        print(f"\n--- Period {p} Goalkeeper Positions ---")
        print(f"  • Home GK Mean X : {h_gk_x:.2f}m  ({'Near Left Goal (X=0)' if h_gk_x < 52.5 else 'Near Right Goal (X=105)'})")
        if pd.notna(a_gk_x):
            print(f"  • Away GK Mean X : {a_gk_x:.2f}m  ({'Near Left Goal (X=0)' if a_gk_x < 52.5 else 'Near Right Goal (X=105)'})")
        
        if h_gk_x < 52.5:
            print(f"  ➔ Period {p} Summary: Home defends Left (X=0) and attacks RIGHT (X=105)")
        else:
            print(f"  ➔ Period {p} Summary: Home defends Right (X=105) and attacks LEFT (X=0)")

print("\n" + "=" * 70)
print("📋 4. EVENT TYPES BREAKDOWN (Full Match)")
print("=" * 70)
print(df_events["Type"].value_counts().to_string())

🔍 1. SPATIAL BOUNDS CHECK (Checking for proper metric conversion)
• Events X Bounds : [-5.25m  to  111.30m]  (Target: 0 to 105m)
• Events Y Bounds : [-2.72m  to  70.72m]  (Target: 0 to 68m)


• Tracking X Bounds: [-4.78m  to  111.30m]  (Target: ~0 to 105m)
• Tracking Y Bounds: [-3.40m  to  71.40m]  (Target: ~0 to 68m)
✅ STATUS: Both Events and Tracking are 100% in METRIC COORDINATES.

⏱️ 2. TEMPORAL & SYNCHRONIZATION CHECK
• Tracking Frame Range: 1 ➔ 145,006 (145,006 frames)
• Events Frame Range  : 1 ➔ 143,630
• Events outside tracking bounds: 0
✅ STATUS: Perfect 1-to-1 temporal frame synchronization.

🧭 3. GOALKEEPER & FIELD ORIENTATION AUDIT (Attacking Direction Ground-Truth)

--- Period 1 Goalkeeper Positions ---
  • Home GK Mean X : 13.07m  (Near Left Goal (X=0))
  • Away GK Mean X : 93.83m  (Near Right Goal (X=105))
  ➔ Period 1 Summary: Home defends Left (X=0) and attacks RIGHT (X=105)

--- Period 2 Goalkeeper Positions ---
  • Home GK Mean X : 95.06m  (Near Right Goal (X=105))
  • Away GK Mean X : 12.26m  (Near Left Goal (X=0))
  ➔ Period 2 Summary: Home defends Right (X=105) and attacks LEFT (X=0)

📋 4. EVENT TYPES BREAKDOWN (Full Match)
Type
PASS              799
R

2

In [168]:
# ==============================================================================
# Verified Tactical Engine: Spatial Corridor Packing & Danger Moments
# Built directly on the Ground-Truth Audit
# ==============================================================================

import numpy as np
import pandas as pd

CORRIDOR_RADIUS_METERS = 4.0  # 4m on each side (8m total corridor width)


def get_attacking_direction(team, period):
    """
    Direction based on audited GK positions:
    - Period 1: Home attacks RIGHT (+1, X=105), Away attacks LEFT (-1, X=0)
    - Period 2: Home attacks LEFT (-1, X=0), Away attacks RIGHT (+1, X=105)
    Returns: (target_goal_x, target_goal_y, attack_direction_sign)
    """
    if period == 1:
        return (105.0, 34.0, 1) if team == "Home" else (0.0, 34.0, -1)
    else:
        return (0.0, 34.0, -1) if team == "Home" else (105.0, 34.0, 1)


def distance_point_to_segment(px, py, x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1
    vector_len_sq = dx * dx + dy * dy

    if vector_len_sq == 0.0:
        return np.hypot(px - x1, py - y1), False

    t = ((px - x1) * dx + (py - y1) * dy) / vector_len_sq

    if 0.0 <= t <= 1.0:
        proj_x = x1 + t * dx
        proj_y = y1 + t * dy
        dist = np.hypot(px - proj_x, py - proj_y)
        return dist, True
    return np.inf, False


def calculate_corridor_packing(event_row, tracking_df):
    start_frame = event_row["Start Frame"]
    team = event_row["Team"]
    period = event_row["Period"]
    x1, y1 = event_row["Start X"], event_row["Start Y"]
    x2, y2 = event_row["End X"], event_row["End Y"]

    if pd.isna(x1) or pd.isna(x2) or pd.isna(y1) or pd.isna(y2):
        return 0

    _, _, dir_sign = get_attacking_direction(team, period)

    # Forward pass condition aligned with attacking direction
    if (x2 - x1) * dir_sign <= 0:
        return 0

    frame_snapshot = tracking_df[tracking_df["Frame"] == start_frame]
    if frame_snapshot.empty:
        return 0

    snapshot_row = frame_snapshot.iloc[0]
    opp_prefix = "Away" if team == "Home" else "Home"
    opp_x_cols = [c for c in tracking_df.columns if c.startswith(opp_prefix) and c.endswith("_x")]

    bypassed_count = 0
    for x_col in opp_x_cols:
        y_col = x_col.replace("_x", "_y")
        def_x = snapshot_row[x_col]
        def_y = snapshot_row[y_col]

        if pd.notna(def_x) and pd.notna(def_y):
            is_between_x = (x1 <= def_x <= x2) if x1 < x2 else (x2 <= def_x <= x1)
            if is_between_x:
                dist, _ = distance_point_to_segment(def_x, def_y, x1, y1, x2, y2)
                if dist <= CORRIDOR_RADIUS_METERS:
                    bypassed_count += 1

    return bypassed_count


def compute_shot_metrics(event_row):
    sx, sy = event_row["Start X"], event_row["Start Y"]
    period = event_row["Period"]
    team = event_row["Team"]

    if pd.isna(sx) or pd.isna(sy):
        return np.nan, np.nan

    goal_x, goal_y, _ = get_attacking_direction(team, period)
    shot_dist = np.hypot(sx - goal_x, sy - goal_y)

    # Posts (7.32m goal width: 30.34m to 37.66m)
    p1 = (goal_x, 37.66)
    p2 = (goal_x, 30.34)
    v1 = np.array([p1[0] - sx, p1[1] - sy])
    v2 = np.array([p2[0] - sx, p2[1] - sy])
    dot = np.dot(v1, v2)
    mag = np.linalg.norm(v1) * np.linalg.norm(v2)
    shot_angle_deg = np.degrees(np.arccos(np.clip(dot / mag, -1.0, 1.0))) if mag > 0 else 0.0

    return round(shot_dist, 2), round(shot_angle_deg, 2)


print("Calculating Corridor Packing and Shot Metrics...")

# 1. Passes
df_passes = df_events[df_events["Type"] == "PASS"].copy()
df_passes["Corridor_Packed"] = df_passes.apply(lambda r: calculate_corridor_packing(r, df_tracking), axis=1)

# 2. Shots
df_shots = df_events[df_events["Type"] == "SHOT"].copy()
shot_metrics = df_shots.apply(compute_shot_metrics, axis=1)
df_shots["Shot_Distance_m"] = [m[0] for m in shot_metrics]
df_shots["Shot_Angle_deg"] = [m[1] for m in shot_metrics]

# 3. High Turnovers (Audited event type: 'RECOVERY')
df_recoveries = df_events[df_events["Type"] == "RECOVERY"].copy()

def is_high_recovery(row):
    sx = row["Start X"]
    if pd.isna(sx):
        return False
    _, _, dir_sign = get_attacking_direction(row["Team"], row["Period"])
    return (sx > 52.5) if dir_sign == 1 else (sx < 52.5)

df_high_regains = df_recoveries[df_recoveries.apply(is_high_recovery, axis=1)].copy()

# Summary Output
df_key_line_breaks = df_passes[df_passes["Corridor_Packed"] >= 2].copy()

print("\n" + "=" * 70)
print("AUDITED TACTICAL SUMMARY:")
print("=" * 70)
print(f"- Total Passes Analyzed: {len(df_passes):,}")
print(f"- Line-Breaking Passes (>= 2 Defenders in Corridor): {len(df_key_line_breaks)}")
print(f"- Line-Breaking Passes (>= 1 Defender in Corridor) : {len(df_passes[df_passes['Corridor_Packed'] >= 1])}")
print(f"- Dangerous High Regains in Opponent Half          : {len(df_high_regains)}")
print(f"- Total Shots Recorded                             : {len(df_shots)}")

print("\n--- Top Line-Breaking Passes in Match ---")
pass_cols = ["Period", "Start Time [s]", "Team", "From", "To", "Corridor_Packed"]
if not df_key_line_breaks.empty:
    print(df_key_line_breaks[pass_cols].sort_values(by="Corridor_Packed", ascending=False).head(8).to_string(index=False))
else:
    print(df_passes[df_passes["Corridor_Packed"] >= 1][pass_cols].head(8).to_string(index=False))

print("\n--- Realistic Match Shots Sample (Distance & Angle) ---")
shot_cols = ["Period", "Start Time [s]", "Team", "From", "Subtype", "Shot_Distance_m", "Shot_Angle_deg"]
print(df_shots[shot_cols].head(6).to_string(index=False))

Calculating Corridor Packing and Shot Metrics...

AUDITED TACTICAL SUMMARY:
- Total Passes Analyzed: 799
- Line-Breaking Passes (>= 2 Defenders in Corridor): 59
- Line-Breaking Passes (>= 1 Defender in Corridor) : 218
- Dangerous High Regains in Opponent Half          : 73
- Total Shots Recorded                             : 24

--- Top Line-Breaking Passes in Match ---
 Period  Start Time [s] Team     From       To  Corridor_Packed
      2         3958.32 Home  Player4  Player5                5
      1         1063.60 Home Player10  Player8                4
      1          730.08 Home  Player3  Player8                3
      2         5187.12 Away Player17 Player18                3
      2         5198.92 Away Player21 Player27                3
      2         3468.12 Away Player21 Player17                3
      2         5355.68 Away Player21 Player17                3
      1         1833.40 Home  Player2  Player9                3

--- Realistic Match Shots Sample (Distance & Angle

In [169]:
# ==============================================================================
# Consolidate Danger Moments (Shots, High Corridor Passes, High Regains)
# ==============================================================================

# 1. High-value Corridor Passes (>= 3 defenders broken or inside final third)
df_danger_passes = df_passes[df_passes["Corridor_Packed"] >= 2].copy()
df_danger_passes["Danger_Category"] = "Line-Breaking Pass"
df_danger_passes["Danger_Value"] = df_danger_passes["Corridor_Packed"]

# 2. Dangerous Shots (Distance <= 20m or on target)
df_danger_shots = df_shots.copy()
df_danger_shots["Danger_Category"] = "Shot"
df_danger_shots["Danger_Value"] = df_danger_shots["Shot_Distance_m"]

# 3. High Turnovers
df_danger_regains = df_high_regains.copy()
df_danger_regains["Danger_Category"] = "High Turnover"
df_danger_regains["Danger_Value"] = 1

# Merge Key Tactical Moments
danger_cols = ["Period", "Start Frame", "Start Time [s]", "Team", "From", "Type", "Danger_Category", "Danger_Value"]
df_danger_moments = pd.concat([
    df_danger_passes[danger_cols],
    df_danger_shots[danger_cols],
    df_danger_regains[danger_cols]
]).sort_values(by="Start Frame").reset_index(drop=True)

print(f"Total Isolated Danger Moments: {len(df_danger_moments)}")
print(df_danger_moments.head(10).to_string(index=False))

Total Isolated Danger Moments: 156
 Period  Start Frame  Start Time [s] Team     From     Type    Danger_Category  Danger_Value
      1          498           19.92 Home  Player2     PASS Line-Breaking Pass          2.00
      1         2289           91.56 Home  Player9     SHOT               Shot          8.64
      1         5228          209.12 Away Player19 RECOVERY      High Turnover          1.00
      1         5549          221.96 Away Player18 RECOVERY      High Turnover          1.00
      1         5923          236.92 Home Player10     SHOT               Shot         22.95
      1         6844          273.76 Home  Player6 RECOVERY      High Turnover          1.00
      1         7203          288.12 Away Player18 RECOVERY      High Turnover          1.00
      1         7404          296.16 Away Player15     PASS Line-Breaking Pass          2.00
      1         7519          300.76 Away Player19 RECOVERY      High Turnover          1.00
      1         7724          308.9

In [170]:
# حفظ اللحظات الخطرة والتمريرات المحسوبة للداشبورد مباشرة
df_danger_moments.to_csv("danger_moments.csv", index=False)
df_passes[df_passes["Corridor_Packed"] >= 1].to_csv("calculated_packing_passes.csv", index=False)
df_shots.to_csv("calculated_shots.csv", index=False)

print("Saved tactical metrics successfully! Ready for Dashboard.")

Saved tactical metrics successfully! Ready for Dashboard.


In [189]:
import numpy as np
import pandas as pd

fps = 25
SHOTS_FILE = "calculated_shots.csv"
PASSES_FILE = "calculated_packing_passes.csv"
OUTPUT_CSV = "filtered_elite_attacks.csv"

# ------------------------------------------------------------------
# 1. تحميل الملفات الأساسية المعتمدة
# ------------------------------------------------------------------
df_shots = pd.read_csv(SHOTS_FILE)
df_packing = pd.read_csv(PASSES_FILE)

candidates = []
goal_frames = set()

# أ. استخراج الأهداف الـ 3 والتصديات الـ 7 من ملف التسديدات
for _, row in df_shots.iterrows():
  f_id = int(row["Start Frame"])
  sub = str(row.get("Subtype", "")).upper()

  if "GOAL" in sub and "GOAL KICK" not in sub:
    goal_frames.add(f_id)
    candidates.append({
        "Period": int(row["Period"]),
        "Start Frame": f_id,
        "Start Time [s]": round(float(f_id) / fps, 2),
        "Team": str(row["Team"]),
        "From": str(row.get("From", "Shooter")),
        "Type": "SHOT",
        "Danger_Category": "Goal Scored",
        "Danger_Value": 10.0,
        "Raw_Danger": 10.0,
        "Outcome_Type": "GOAL",
        "Outcome_Desc": f"Goal Scored ({row.get('From', 'Shooter')})",
    })
  elif "SAVED" in sub:
    candidates.append({
        "Period": int(row["Period"]),
        "Start Frame": f_id,
        "Start Time [s]": round(float(f_id) / fps, 2),
        "Team": str(row["Team"]),
        "From": str(row.get("From", "Shooter")),
        "Type": "SHOT",
        "Danger_Category": "Big Chance Saved",
        "Danger_Value": 8.0,
        "Raw_Danger": 8.0,
        "Outcome_Type": "SAVED",
        "Outcome_Desc": "Shot Saved by GK",
    })

# ب. استخراج التمريرات الكاسرة للخطوط (Corridor_Packed >= 3)
key_passes = df_packing[df_packing["Corridor_Packed"] >= 3].copy()
for _, row in key_passes.iterrows():
  f_id = int(row["Start Frame"])
  candidates.append({
      "Period": int(row["Period"]),
      "Start Frame": f_id,
      "Start Time [s]": round(float(f_id) / fps, 2),
      "Team": str(row["Team"]),
      "From": str(row.get("From", "Passer")),
      "Type": "PASS",
      "Danger_Category": "Line-Breaking Pass",
      "Danger_Value": float(row["Corridor_Packed"]),
      "Raw_Danger": float(row["Corridor_Packed"]),
      "Outcome_Type": "PROGRESSION",
      "Outcome_Desc": "Attack Progression",
  })

df_candidates = pd.DataFrame(candidates)
df_candidates = (
    df_candidates.drop_duplicates(subset=["Period", "Start Frame", "Team"])
    .sort_values(by=["Period", "Start Frame"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------------
# 2. تتبع دورة حياة الهجمة والـ Attack Chain (بدون تكرار أعمدة)
# ------------------------------------------------------------------
def trace_possession_lifecycle(row, events_df):
  target_frame = int(row["Start Frame"])
  period = int(row["Period"])
  team = str(row["Team"])
  cat = str(row["Danger_Category"])

  p_events = (
      events_df[events_df["Period"] == period]
      .sort_values("Start Frame")
      .reset_index(drop=True)
  )
  matches = p_events.index[p_events["Start Frame"] == target_frame].tolist()
  curr_idx = (
      matches[0]
      if matches
      else (p_events[p_events["Team"] == team]["Start Frame"] - target_frame)
      .abs()
      .idxmin()
  )

  # تتبع للخلف (حتى 15 ثانية)
  start_idx = curr_idx
  back_limit = target_frame - (15 * fps)
  for i in range(curr_idx - 1, -1, -1):
    r = p_events.loc[i]
    if r["Start Frame"] < back_limit:
      break
    if str(r["Team"]) != team and "PASS" in str(r.get("Type", "")).upper():
      start_idx = i + 1
      break
    else:
      start_idx = i
      if any(
          k in str(r.get("Type", "")).upper()
          for k in ["RECOVERY", "FREE KICK", "CORNER"]
      ):
        break

  # تتبع للأمام (حتى 15 ثانية)
  end_idx = curr_idx
  fwd_limit = target_frame + (15 * fps)

  # نأخذ القيمة المبدئية كنص صريح
  outcome_type = str(row.get("Outcome_Type", "PROGRESSION"))
  outcome_desc = str(row.get("Outcome_Desc", "Attack Progression"))

  if outcome_type != "GOAL":
    for j in range(curr_idx, min(len(p_events), curr_idx + 25)):
      r = p_events.loc[j]
      if r["Start Frame"] > fwd_limit:
        break
      r_type = str(r.get("Type", "")).upper()
      r_sub = str(r.get("Subtype", "")).upper()
      r_from = str(r.get("From", "Player"))
      end_idx = j

      if "GOAL" in r_sub and "GOAL KICK" not in r_sub:
        outcome_type = "GOAL"
        outcome_desc = f"Goal Scored ({r_from})"
        break
      elif "SHOT" in r_type:
        nxt_j = j + 1
        if (
            nxt_j < len(p_events)
            and "GOAL" in str(p_events.loc[nxt_j].get("Subtype", "")).upper()
        ):
          outcome_type = "GOAL"
          outcome_desc = f"Goal Scored ({r_from})"
          end_idx = nxt_j
          break
        elif "SAVED" in r_sub:
          outcome_type = "SAVED"
          outcome_desc = "Shot Saved by GK"
        elif "BLOCKED" in r_sub:
          outcome_type = "BLOCKED"
          outcome_desc = "Shot Blocked by Defense"
        elif "OFF TARGET" in r_sub or "WIDE" in r_sub:
          outcome_type = "OFF_TARGET"
          outcome_desc = "Shot Off Target"
        else:
          outcome_type = "SHOT"
          outcome_desc = f"Shot on Goal ({r_from})"
        break
      elif any(k in r_type for k in ["OUT", "BALL OUT"]):
        outcome_type = "OUT"
        outcome_desc = "Ball Out of Play"
        break
      elif "OFFSIDE" in r_type:
        outcome_type = "STOPPAGE"
        outcome_desc = "Offside Whistle"
        break
      elif str(r["Team"]) != team and "PASS" in r_type:
        outcome_type = "TURNOVER"
        outcome_desc = "Ball Intercepted"
        break

  seq_start = int(p_events.loc[start_idx, "Start Frame"])
  raw_end = int(
      p_events.loc[end_idx, "End Frame"]
      if pd.notnull(p_events.loc[end_idx].get("End Frame"))
      else p_events.loc[end_idx, "Start Frame"] + 40
  )
  seq_end = raw_end + 125

  steps = []
  for _, s in p_events.loc[start_idx:end_idx].iterrows():
    p = str(s.get("From", ""))
    t = str(s.get("Type", "")).upper()
    sub = str(s.get("Subtype", "")).upper()
    if not p or p == "nan":
      continue
    if "RECOVERY" in t or "RECOVERY" in sub:
      steps.append(f"Recovery ({p})")
    elif "PASS" in t:
      steps.append(f"Pass ({p})")
    elif "SHOT" in t:
      if "GOAL" in sub and "GOAL KICK" not in sub:
        steps.append(f"Goal ⚽ ({p})")
      else:
        steps.append(f"Shot 🎯 ({p})")

  clean_steps = []
  for stp in steps:
    if not clean_steps or stp != clean_steps[-1]:
      clean_steps.append(stp)
  chain_str = (
      " ➔ ".join(clean_steps)
      if clean_steps
      else f"Build-up ➔ {cat} ({row['From']})"
  )

  return pd.Series({
      "Seq_Start_Frame": seq_start,
      "Key_Action_Frame": target_frame,
      "Seq_End_Frame": seq_end,
      "Seq_Duration_s": round((seq_end - seq_start) / fps, 2),
      "Updated_Outcome_Type": outcome_type,
      "Updated_Outcome_Desc": outcome_desc,
      "Attack_Chain": chain_str,
  })


seq_meta = df_candidates.apply(
    lambda r: trace_possession_lifecycle(r, df_events), axis=1
)

# دمج الأعمدة واستبدال المخرجات دون تكرار أسماء الأعمدة
df_all = pd.concat([df_candidates, seq_meta], axis=1)
df_all["Outcome_Type"] = df_all["Updated_Outcome_Type"]
df_all["Outcome_Desc"] = df_all["Updated_Outcome_Desc"]
df_all = df_all.drop(
    columns=["Updated_Outcome_Type", "Updated_Outcome_Desc"]
).copy()

# التأكد الصارم من عدم وجود أي أعمدة مكررة
df_all = df_all.loc[:, ~df_all.columns.duplicated()].copy()

# ------------------------------------------------------------------
# 3. الدمج التكتيكي لمنع التكرار الزمني
# ------------------------------------------------------------------
df_all = df_all.sort_values(by=["Period", "Start Frame"]).reset_index(drop=True)

merged = []
skip = set()

for i in range(len(df_all)):
  if i in skip:
    continue
  curr = df_all.iloc[i].to_dict()

  j = i + 1
  while j < len(df_all):
    nxt = df_all.iloc[j].to_dict()
    same_team = (curr["Period"] == nxt["Period"]) and (
        curr["Team"] == nxt["Team"]
    )
    time_diff = (nxt["Start Frame"] - curr["Start Frame"]) / fps

    # دمج الأحداث لنفس الفريق خلال 4.5 ثوانٍ
    if same_team and time_diff <= 4.5:
      skip.add(j)
      curr["Seq_End_Frame"] = max(curr["Seq_End_Frame"], nxt["Seq_End_Frame"])
      curr["Seq_Duration_s"] = round(
          (curr["Seq_End_Frame"] - curr["Seq_Start_Frame"]) / fps, 2
      )

      # الحفاظ على هوية التمريرة الكاسرة للخطوط كأولوية تكتيكية
      if (
          curr["Danger_Category"] != "Line-Breaking Pass"
          and nxt["Danger_Category"] == "Line-Breaking Pass"
      ):
        curr["Danger_Category"] = "Line-Breaking Pass"
        curr["Danger_Value"] = nxt["Danger_Value"]
        curr["From"] = nxt["From"]
        curr["Key_Action_Frame"] = nxt["Start Frame"]
        curr["Start Time [s]"] = nxt["Start Time [s]"]

      # قفل النتيجة للأهداف
      c_out = str(curr.get("Outcome_Type", ""))
      n_out = str(nxt.get("Outcome_Type", ""))

      if n_out == "GOAL" or c_out == "GOAL":
        curr["Outcome_Type"] = "GOAL"
        curr["Outcome_Desc"] = (
            nxt["Outcome_Desc"] if n_out == "GOAL" else curr["Outcome_Desc"]
        )
      elif c_out in ["PROGRESSION", "OFF_TARGET"]:
        curr["Outcome_Type"] = nxt["Outcome_Type"]
        curr["Outcome_Desc"] = nxt["Outcome_Desc"]

      j += 1
    else:
      break

  # تصحيح تكتيكي لهجمة الدقيقة 66 (سلسلة 98958)
  if curr["Period"] == 2 and abs(curr["Start Frame"] - 98958) < 150:
    curr["Danger_Category"] = "Line-Breaking Pass"
    curr["Outcome_Type"] = "GOAL"
    curr["Outcome_Desc"] = "Goal Scored (Player9 after Rebound)"
    curr["From"] = "Player4"
    curr["Danger_Value"] = 5.0

  merged.append(curr)

df_final = pd.DataFrame(merged)
export_cols = [
    "Period",
    "Start Frame",
    "Start Time [s]",
    "Team",
    "From",
    "Type",
    "Danger_Category",
    "Danger_Value",
    "Seq_Start_Frame",
    "Key_Action_Frame",
    "Seq_End_Frame",
    "Seq_Duration_s",
    "Outcome_Type",
    "Outcome_Desc",
    "Attack_Chain",
]
df_final = df_final[export_cols]
df_final.to_csv(OUTPUT_CSV, index=False)

print("=" * 70)
print(
    f"✅ تم بنجاح إنشاء ملف الهجمات النخبوية المعزول: '{OUTPUT_CSV}'"
    f" ({len(df_final)} هجمة)"
)
print("=" * 70)
print("\n📌 توزيع فئات الخطر (Danger_Category):")
print(df_final["Danger_Category"].value_counts())
print("\n🎯 توزيع النتائج النهائية (Outcome_Type):")
print(df_final["Outcome_Type"].value_counts())
print("\n📋 ملخص الهجمات المستخرجة:")
print(
    df_final[[
        "Period",
        "Start Time [s]",
        "Team",
        "From",
        "Danger_Category",
        "Danger_Value",
        "Outcome_Type",
    ]].to_string(index=False)
)

✅ تم بنجاح إنشاء ملف الهجمات النخبوية المعزول: 'filtered_elite_attacks.csv' (16 هجمة)

📌 توزيع فئات الخطر (Danger_Category):
Danger_Category
Line-Breaking Pass    8
Big Chance Saved      6
Goal Scored           2
Name: count, dtype: int64

🎯 توزيع النتائج النهائية (Outcome_Type):
Outcome_Type
SAVED          6
GOAL           3
OFF_TARGET     3
OUT            2
TURNOVER       1
PROGRESSION    1
Name: count, dtype: int64

📋 ملخص الهجمات المستخرجة:
 Period  Start Time [s] Team     From    Danger_Category  Danger_Value Outcome_Type
      1           91.56 Home  Player9        Goal Scored          10.0         GOAL
      1          385.12 Home  Player8   Big Chance Saved           8.0        SAVED
      1          730.08 Home  Player3 Line-Breaking Pass           3.0   OFF_TARGET
      1          778.80 Away Player24   Big Chance Saved           8.0        SAVED
      1         1063.60 Home Player10 Line-Breaking Pass           4.0   OFF_TARGET
      1         1833.40 Home  Player2 Line-Brea

In [ ]:
import os
import numpy as np
import pandas as pd

# 1. إعداد البيانات وترتيبها
PASSES_FILE = "calculated_packing_passes.csv"
fps = 25

df_events = df_events.sort_values(by=["Period", "Start Frame"]).reset_index(
    drop=True
)
df_passes = (
    pd.read_csv(PASSES_FILE)
    if (os.path.exists(PASSES_FILE) or pd.io.common.file_exists(PASSES_FILE))
    else None
)

# ------------------------------------------------------------------
# الخطوة 1: استخراج الفئات بترتيب الأولوية (أهداف -> شوطات خطيرة -> باصات)
# ------------------------------------------------------------------
candidate_moments = []

# أ. استخراج الأهداف أولاً في حتة لوحدها (الأولوية القصوى)
shots_all = df_events[
    df_events["Type"].astype(str).str.upper().str.contains("SHOT")
].copy()
goals_df = shots_all[
    shots_all["Subtype"].astype(str).str.upper().str.contains("GOAL", na=False)
].copy()

goal_frames = set()
for _, row in goals_df.iterrows():
  f_id = int(row["Start Frame"])
  goal_frames.add(f_id)
  candidate_moments.append({
      "Period": int(row["Period"]),
      "Start Frame": f_id,
      "Start Time [s]": round(float(f_id) / fps, 2),
      "Team": str(row["Team"]),
      "From": str(row.get("From", "Shooter")),
      "Type": "SHOT",
      "Danger_Category": "Goal Scored",
      "Raw_Danger": 10.0,
  })

print(f"🎯 تم تثبيت {len(goals_df)} هدف صريح في المباراة.")

# ب. أهم الشوطات الخطيرة فقط (SAVED / BLOCKED) بدون تكرار الأهداف
for _, row in shots_all.iterrows():
  f_id = int(row["Start Frame"])
  if f_id in goal_frames:
    continue

  sub = str(row.get("Subtype", "")).upper()
  if "SAVED" in sub:
    candidate_moments.append({
        "Period": int(row["Period"]),
        "Start Frame": f_id,
        "Start Time [s]": round(float(f_id) / fps, 2),
        "Team": str(row["Team"]),
        "From": str(row.get("From", "Shooter")),
        "Type": "SHOT",
        "Danger_Category": "Big Chance Saved",
        "Raw_Danger": 8.0,
    })
  elif "BLOCKED" in sub:
    candidate_moments.append({
        "Period": int(row["Period"]),
        "Start Frame": f_id,
        "Start Time [s]": round(float(f_id) / fps, 2),
        "Team": str(row["Team"]),
        "From": str(row.get("From", "Shooter")),
        "Type": "SHOT",
        "Danger_Category": "Blocked Shot",
        "Raw_Danger": 6.5,
    })

# ج. التمريرات النوعية الكاسرة لخطوط الدفاع (Line-Breaking Passes)
if df_passes is not None:
  pack_col = None
  for col in [
      "Packed_Defenders",
      "packed_defenders",
      "Packed",
      "packed",
      "Danger_Value",
  ]:
    if col in df_passes.columns:
      pack_col = col
      break

  if pack_col:
    key_passes = df_passes[df_passes[pack_col] >= 3].copy()
    for _, row in key_passes.iterrows():
      candidate_moments.append({
          "Period": int(row["Period"]),
          "Start Frame": int(row["Start Frame"]),
          "Start Time [s]": round(float(row["Start Frame"]) / fps, 2),
          "Team": str(row["Team"]),
          "From": str(row.get("From", "Passer")),
          "Type": "PASS",
          "Danger_Category": "Line-Breaking Pass",
          "Raw_Danger": float(row[pack_col]),
      })
    print(
        f"✅ تم تحميل {len(key_passes)} تمريرة كاسرة للخطوط من عمود '{pack_col}'"
    )

df_candidates = pd.DataFrame(candidate_moments)
df_candidates = (
    df_candidates.drop_duplicates(subset=["Period", "Start Frame", "Team"])
    .sort_values(by=["Period", "Start Frame"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------------
# الخطوة 2: تتبع مسار وبناء كل هجمة للخلف وللأمام
# ------------------------------------------------------------------
def trace_possession_lifecycle(row, events_df):
  target_frame = int(row["Start Frame"])
  period = int(row["Period"])
  team = str(row["Team"])
  cat = str(row["Danger_Category"])

  p_events = (
      events_df[events_df["Period"] == period]
      .sort_values("Start Frame")
      .reset_index(drop=True)
  )

  matches = p_events.index[p_events["Start Frame"] == target_frame].tolist()
  curr_idx = (
      matches[0]
      if matches
      else (p_events[p_events["Team"] == team]["Start Frame"] - target_frame)
      .abs()
      .idxmin()
  )

  # تتبع بداية الهجمة للخلف
  start_idx = curr_idx
  back_limit = target_frame - (15 * fps)
  for i in range(curr_idx - 1, -1, -1):
    r = p_events.loc[i]
    if r["Start Frame"] < back_limit:
      break
    if str(r["Team"]) != team and "PASS" in str(r.get("Type", "")).upper():
      start_idx = i + 1
      break
    else:
      start_idx = i
      if any(
          k in str(r.get("Type", "")).upper()
          for k in ["RECOVERY", "FREE KICK", "CORNER"]
      ):
        break

  # تتبع نهاية الهجمة للأمام
  end_idx = curr_idx
  fwd_limit = target_frame + (15 * fps)

  # لو الحدث مرشح كهدف صريح يبقى نهايته حتماً GOAL
  if cat == "Goal Scored":
    outcome_type = "GOAL"
    outcome_desc = f"Goal Scored ({row['From']})"
  else:
    outcome_type = "PROGRESSION"
    outcome_desc = "Attack Progression"

  for j in range(curr_idx, min(len(p_events), curr_idx + 25)):
    r = p_events.loc[j]
    if r["Start Frame"] > fwd_limit:
      break
    r_type = str(r.get("Type", "")).upper()
    r_sub = str(r.get("Subtype", "")).upper()
    r_from = str(r.get("From", "Player"))
    end_idx = j

    if "GOAL" in r_sub and "GOAL KICK" not in r_sub:
      outcome_type = "GOAL"
      outcome_desc = f"Goal Scored ({r_from})"
      break
    elif "SHOT" in r_type:
      nxt_j = j + 1
      if (
          nxt_j < len(p_events)
          and "GOAL" in str(p_events.loc[nxt_j].get("Subtype", "")).upper()
      ):
        outcome_type = "GOAL"
        outcome_desc = f"Goal Scored ({r_from})"
        end_idx = nxt_j
        break
      elif "SAVED" in r_sub:
        outcome_type = "SAVED"
        outcome_desc = "Shot Saved by GK"
      elif "BLOCKED" in r_sub:
        outcome_type = "BLOCKED"
        outcome_desc = "Shot Blocked by Defense"
      elif "OFF TARGET" in r_sub or "WIDE" in r_sub:
        outcome_type = "OFF_TARGET"
        outcome_desc = "Shot Off Target"
      else:
        outcome_type = "SHOT"
        outcome_desc = f"Shot on Goal ({r_from})"
      break
    elif any(k in r_type for k in ["OUT", "BALL OUT"]):
      outcome_type = "OUT"
      outcome_desc = "Ball Out of Play"
      break
    elif "OFFSIDE" in r_type:
      outcome_type = "STOPPAGE"
      outcome_desc = "Offside Whistle"
      break
    elif str(r["Team"]) != team and "PASS" in r_type:
      outcome_type = "TURNOVER"
      outcome_desc = "Ball Intercepted"
      break

  seq_start = int(p_events.loc[start_idx, "Start Frame"])
  raw_end = int(
      p_events.loc[end_idx, "End Frame"]
      if pd.notnull(p_events.loc[end_idx].get("End Frame"))
      else p_events.loc[end_idx, "Start Frame"] + 40
  )
  seq_end = raw_end + 125

  # تنظيف خطوات السلسلة
  steps = []
  for _, s in p_events.loc[start_idx:end_idx].iterrows():
    p = str(s.get("From", ""))
    t = str(s.get("Type", "")).upper()
    sub = str(s.get("Subtype", "")).upper()
    if not p or p == "nan":
      continue
    if "RECOVERY" in t or "RECOVERY" in sub:
      steps.append(f"Recovery ({p})")
    elif "PASS" in t:
      steps.append(f"Pass ({p})")
    elif "SHOT" in t:
      if "GOAL" in sub and "GOAL KICK" not in sub:
        steps.append(f"Goal ⚽ ({p})")
      else:
        steps.append(f"Shot 🎯 ({p})")

  clean_steps = []
  for stp in steps:
    if not clean_steps or stp != clean_steps[-1]:
      clean_steps.append(stp)
  chain_str = (
      " ➔ ".join(clean_steps)
      if clean_steps
      else f"Build-up ➔ {cat} ({row['From']})"
  )

  return pd.Series({
      "Seq_Start_Frame": seq_start,
      "Key_Action_Frame": target_frame,
      "Seq_End_Frame": seq_end,
      "Seq_Duration_s": round((seq_end - seq_start) / fps, 2),
      "Outcome_Type": outcome_type,
      "Outcome_Desc": outcome_desc,
      "Attack_Chain": chain_str,
  })


seq_metadata = df_candidates.apply(
    lambda r: trace_possession_lifecycle(r, df_events), axis=1
)
df_all = pd.concat([df_candidates, seq_metadata], axis=1)

# ------------------------------------------------------------------
# الخطوة 3: الدمج التكتيكي الذكي (الحفاظ على هوية التمريرة كبطل للهجمة)
# ------------------------------------------------------------------
df_all = df_all.sort_values(by=["Period", "Start Frame"]).reset_index(drop=True)

merged_attacks = []
skip_set = set()

for i in range(len(df_all)):
  if i in skip_set:
    continue
  curr = df_all.iloc[i].to_dict()

  j = i + 1
  while j < len(df_all):
    nxt = df_all.iloc[j]
    same_team = curr["Period"] == nxt["Period"] and curr["Team"] == nxt["Team"]

    # دمج الأحداث لنفس الفريق خلال تداخل فريمات الهجمة
    if same_team and nxt["Start Frame"] <= curr["Seq_End_Frame"] + 50:
      skip_set.add(j)
      curr["Seq_End_Frame"] = max(curr["Seq_End_Frame"], nxt["Seq_End_Frame"])
      curr["Seq_Duration_s"] = round(
          (curr["Seq_End_Frame"] - curr["Seq_Start_Frame"]) / fps, 2
      )

      # لو الحدث المدمج هدف تتقفل النتيجة هدفاً فوراً
      if nxt["Outcome_Type"] == "GOAL":
        curr["Outcome_Type"] = "GOAL"
        curr["Outcome_Desc"] = nxt["Outcome_Desc"]

      # إذا كانت الهجمة فيها باص كاسر للخطوط يظل هو الفئة التكتيكية الأساسية
      if curr["Danger_Category"] != "Line-Breaking Pass":
        if nxt["Danger_Category"] == "Line-Breaking Pass":
          curr["Danger_Category"] = "Line-Breaking Pass"
          curr["Key_Action_Frame"] = nxt["Start Frame"]
          curr["From"] = nxt["From"]
          curr["Raw_Danger"] = nxt["Raw_Danger"]

      j += 1
    else:
      break

  merged_attacks.append(curr)

df_final_elite = pd.DataFrame(merged_attacks)
df_final_elite["Danger_Value"] = df_final_elite["Raw_Danger"]

# حفظ الملف
OUTPUT_NAME = "elite_sequences_18.csv"
df_final_elite.to_csv(OUTPUT_NAME, index=False)

print(
    f"\n✅ تم استخراج وحفظ {len(df_final_elite)} هجمة فريدة بالمعايير التكتيكية"
    f" الكاملة في '{OUTPUT_NAME}'."
)
print("\nتوزيع فئات الهجمات:")
print(df_final_elite["Danger_Category"].value_counts())
print("\nتوزيع النتائج النهائية:")
print(df_final_elite["Outcome_Type"].value_counts())

🎯 تم تثبيت 3 هدف صريح في المباراة.

✅ تم استخراج وحفظ 9 هجمة فريدة بالمعايير التكتيكية الكاملة في 'elite_sequences_18.csv'.

توزيع فئات الهجمات:
Danger_Category
Big Chance Saved    7
Goal Scored         2
Name: count, dtype: int64

توزيع النتائج النهائية:
Outcome_Type
SAVED    6
GOAL     3
Name: count, dtype: int64


In [190]:
import json
import numpy as np
import pandas as pd

INPUT_CSV = "filtered_elite_attacks.csv"
OUTPUT_JSON = "elite_attacks_catalog.json"

df = pd.read_csv(INPUT_CSV)

attacks_catalog = {}

for _, row in df.iterrows():
  # مفتاح فريد لكل هجمة مبني على الفريم والوقت
  frame_key = f"frame_{int(row['Start Frame'])}"

  # تنسيق الدقيقة والثانية
  total_s = float(row["Start Time [s]"])
  minutes = int(total_s // 60)
  seconds = int(total_s % 60)
  time_display = f"{minutes:02d}:{seconds:02d}"

  attacks_catalog[frame_key] = {
      "metadata": {
          "period": int(row["Period"]),
          "start_frame": int(row["Start Frame"]),
          "start_time_s": round(total_s, 2),
          "time_display": time_display,
          "team": str(row["Team"]),
          "primary_player": str(row["From"]),
          "action_type": str(row["Type"]),
      },
      "tactical_classification": {
          "danger_category": str(row["Danger_Category"]),
          "danger_value": float(row["Danger_Value"]),
          "outcome_type": str(row["Outcome_Type"]),
          "outcome_desc": str(row["Outcome_Desc"]),
      },
      "sequence_tracking": {
          "seq_start_frame": int(row["Seq_Start_Frame"]),
          "key_action_frame": int(row["Key_Action_Frame"]),
          "seq_end_frame": int(row["Seq_End_Frame"]),
          "duration_s": float(row["Seq_Duration_s"]),
          "attack_chain": str(row["Attack_Chain"]),
      },
      # الحقول دي جاهزة عشان الـ LLM يملأ فيها تحليله مباشرة
      "llm_tactical_evaluation": {
          "status": "pending",
          "tactical_summary": None,
          "space_exploited": None,
          "defensive_breakdown": None,
      },
  }

# حفظ الملف بتنسيق JSON مرتب وقابل للقراءة
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
  json.dump(attacks_catalog, f, indent=4, ensure_ascii=False)

print(
    f"✅ تم تحويل الـ 16 هجمة بنجاح إلى ملف JSON منظم: '{OUTPUT_JSON}'"
)
print(f"🔑 إجمالي المفاتيح المسجلة: {len(attacks_catalog)}")

# طباعة عينة من أول هجمة للتأكد من الهيكل
first_key = list(attacks_catalog.keys())[0]
print(f"\n🔍 عينة لهيكل الهجمة الأولى ({first_key}):")
print(json.dumps(attacks_catalog[first_key], indent=2, ensure_ascii=False))

✅ تم تحويل الـ 16 هجمة بنجاح إلى ملف JSON منظم: 'elite_attacks_catalog.json'
🔑 إجمالي المفاتيح المسجلة: 16

🔍 عينة لهيكل الهجمة الأولى (frame_2289):
{
  "metadata": {
    "period": 1,
    "start_frame": 2289,
    "start_time_s": 91.56,
    "time_display": "01:31",
    "team": "Home",
    "primary_player": "Player9",
    "action_type": "SHOT"
  },
  "tactical_classification": {
    "danger_category": "Goal Scored",
    "danger_value": 10.0,
    "outcome_type": "GOAL",
    "outcome_desc": "Goal Scored (Player9)"
  },
  "sequence_tracking": {
    "seq_start_frame": 2143,
    "key_action_frame": 2289,
    "seq_end_frame": 2434,
    "duration_s": 11.64,
    "attack_chain": "Pass (Player6) ➔ Pass (Player10) ➔ Goal ⚽ (Player9)"
  },
  "llm_tactical_evaluation": {
    "status": "pending",
    "tactical_summary": null,
    "space_exploited": null,
    "defensive_breakdown": null
  }
}


In [202]:
import pandas as pd

# الملفات المراد تحديثها وتثبيت التعديل فيها
target_csv_files = [
    "filtered_elite_attacks.csv",
    "elite_sequences_16_ready.csv"
]

for file_path in target_csv_files:
    try:
        df = pd.read_csv(file_path)
        
        # استهداف السطر الخاص بهجمة الدقيقة 66 (Player4 / ثانية 3958.32)
        mask = (df["Period"] == 2) & (abs(df["Start Time [s]"] - 3958.32) < 2.0)
        
        if mask.any():
            df.loc[mask, "Outcome_Type"] = "GOAL"
            df.loc[mask, "Outcome_Desc"] = "Goal Scored (Player9 after Rebound)"
            
            if "Attack_Chain" in df.columns:
                df.loc[mask, "Attack_Chain"] = (
                    "Pass (Player8) ➔ Line-Break Pass (Player4) ➔ Shot (Player5) ➔ Goal ⚽ (Player9)"
                )
            
            # حفظ التعديل في نفس الملف
            df.to_csv(file_path, index=False)
            print(f"✅ تم تحديث وتثبيت التعديلات في الملف: '{file_path}'")
        else:
            print(f"⚠️ لم يتم العثور على سطر الهجمة في الملف: '{file_path}'")
            
    except FileNotFoundError:
        print(f"❌ الملف غير موجود: '{file_path}'")

# طباعة التأكيد النهائي من أحد الملفات
df_check = pd.read_csv("elite_sequences_16_ready.csv")
print("\n" + "=" * 65)
print("🎯 توزيع النتائج النهائي المؤكد في ملفات الـ CSV:")
print(df_check["Outcome_Type"].value_counts().to_string())
print("=" * 65)
print("\n📋 الأهداف الثلاثة المؤكدة في الـ CSV:")
print(
    df_check[df_check["Outcome_Type"] == "GOAL"][[
        "Start Time [s]", "Team", "From", "Danger_Category", "Outcome_Type", "Outcome_Desc"
    ]].to_string(index=False)
)

✅ تم تحديث وتثبيت التعديلات في الملف: 'filtered_elite_attacks.csv'
✅ تم تحديث وتثبيت التعديلات في الملف: 'elite_sequences_16_ready.csv'

🎯 توزيع النتائج النهائي المؤكد في ملفات الـ CSV:
Outcome_Type
SAVED          6
GOAL           3
OFF_TARGET     3
OUT            2
TURNOVER       1
PROGRESSION    1

📋 الأهداف الثلاثة المؤكدة في الـ CSV:
 Start Time [s] Team     From    Danger_Category Outcome_Type                        Outcome_Desc
          91.56 Home  Player9        Goal Scored         GOAL               Goal Scored (Player9)
        3600.20 Home Player10        Goal Scored         GOAL              Goal Scored (Player10)
        3958.32 Home  Player4 Line-Breaking Pass         GOAL Goal Scored (Player9 after Rebound)


The Analyst Room

In [3]:
import os
os.environ["GROQ_API_KEY"] = "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"  

    
import os
import json
import time
import pandas as pd
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
SELECTED_MODEL = "qwen/qwen3.8-27b"
OUTPUT_FILE = "top_analyst_events.json"

In [206]:
import pandas as pd
import numpy as np

fps = 25

# 1. قراءة ملف الهجمات النخبوية المصفى
INPUT_CSV = "filtered_elite_attacks.csv"
df_elite = pd.read_csv(INPUT_CSV)

# التأكد من ترتيب جدول الأحداث
df_events = df_events.sort_values(by=["Period", "Start Frame"]).reset_index(drop=True)

def trace_attack_sequence_robust(event_row, df_events_all, max_lookback_sec=15, max_lookforward_sec=12):
    target_frame = int(event_row["Start Frame"])
    period = int(event_row["Period"])
    team = str(event_row["Team"])
    
    # أحداث نفس الشوط
    period_events = df_events_all[df_events_all["Period"] == period].sort_values("Start Frame").reset_index(drop=True)
    
    # تحديد موقع الحدث الحالي
    event_idx_list = period_events.index[period_events["Start Frame"] == target_frame].tolist()
    if not event_idx_list:
        team_events = period_events[period_events["Team"] == team]
        curr_idx = (team_events["Start Frame"] - target_frame).abs().idxmin() if not team_events.empty else 0
    else:
        curr_idx = event_idx_list[0]
        
    max_back_frames = max_lookback_sec * fps
    max_fwd_frames = max_lookforward_sec * fps
    
    # --- 1. التتبع للخلف ---
    start_idx = curr_idx
    back_limit_frame = target_frame - max_back_frames
    
    for i in range(curr_idx - 1, -1, -1):
        prev_row = period_events.loc[i]
        if prev_row["Start Frame"] < back_limit_frame or prev_row["Period"] != period:
            break
        prev_team = str(prev_row["Team"])
        prev_type = str(prev_row.get("Type", "")).upper()
        prev_sub = str(prev_row.get("Subtype", "")).upper()
        
        if prev_team != team:
            if prev_type == "PASS":
                start_idx = i + 1
                break
        else:
            start_idx = i
            if any(k in prev_type or k in prev_sub for k in ["RECOVERY", "TACKLE", "INTERCEPTION", "BALL OUT", "FREE KICK", "CORNER"]):
                break

    # --- 2. التتبع للأمام وكشف الهدف حتى لو بعد ارتداد ---
    end_idx = curr_idx
    fwd_limit_frame = target_frame + max_fwd_frames
    
    outcome_desc = "Attack Progression"
    outcome_type = "PROGRESSION"
    
    # فحص مسبق: هل في خلال نافذة الهجمة للأمام حدث أي هدف لنفس الفريق؟
    future_window = period_events[
        (period_events.index >= curr_idx) & 
        (period_events["Start Frame"] <= fwd_limit_frame)
    ]
    
    goal_in_window = future_window[
        (future_window["Team"] == team) & 
        (future_window["Subtype"].astype(str).str.upper().str.contains("GOAL", na=False)) & 
        (~future_window["Subtype"].astype(str).str.upper().str.contains("GOAL KICK", na=False))
    ]
    
    has_subsequent_goal = not goal_in_window.empty
    
    for j in range(curr_idx, min(len(period_events), curr_idx + 25)):
        nxt_row = period_events.loc[j]
        if nxt_row["Start Frame"] > fwd_limit_frame or nxt_row["Period"] != period:
            break
            
        nxt_team = str(nxt_row["Team"])
        nxt_type = str(nxt_row.get("Type", "")).upper()
        nxt_sub = str(nxt_row.get("Subtype", "")).upper()
        nxt_from = str(nxt_row.get("From", "Player"))
        end_idx = j
        
        if "GOAL" in nxt_sub and "GOAL KICK" not in nxt_sub:
            outcome_desc = f"Goal Scored ({nxt_from})"
            outcome_type = "GOAL"
            break
            
        elif "SHOT" in nxt_type:
            # لو في هدف تالي في نفس الهجمة (ارتداد)، لا نقفل النتيجة كـ SAVED أو BLOCKED بل نستمر للهدف
            if has_subsequent_goal:
                continue
                
            if "SAVED" in nxt_sub:
                outcome_desc = "Shot Saved by GK"
                outcome_type = "SAVED"
            elif "BLOCKED" in nxt_sub:
                outcome_desc = "Shot Blocked by Defense"
                outcome_type = "BLOCKED"
            elif "OFF TARGET" in nxt_sub or "WIDE" in nxt_sub:
                outcome_desc = "Shot Off Target"
                outcome_type = "OFF_TARGET"
            else:
                outcome_desc = f"Shot on Goal ({nxt_from})"
                outcome_type = "SHOT"
            break
            
        elif any(k in nxt_type or k in nxt_sub for k in ["BALL OUT", "OUT"]):
            if not has_subsequent_goal:
                outcome_desc = "Ball Out of Play"
                outcome_type = "OUT"
                break
                
        elif nxt_team != team and nxt_type == "PASS":
            if not has_subsequent_goal:
                outcome_desc = "Possession Lost / Intercepted"
                outcome_type = "TURNOVER"
                break

    # استخراج الفريمات
    seq_start_frame = int(period_events.loc[start_idx, "Start Frame"])
    raw_end = int(period_events.loc[end_idx, "End Frame"] if pd.notnull(period_events.loc[end_idx].get("End Frame")) else period_events.loc[end_idx, "Start Frame"] + 40)
    seq_end_frame = raw_end + 25
    
    # بناء الـ Attack Chain
    chain_slice = period_events.loc[start_idx:end_idx]
    steps = []
    for _, s_row in chain_slice.iterrows():
        st_from = str(s_row.get("From", ""))
        st_type = str(s_row.get("Type", "")).upper()
        st_sub = str(s_row.get("Subtype", "")).upper()
        if not st_from or st_from == "nan":
            continue
        if "RECOVERY" in st_type or "RECOVERY" in st_sub:
            steps.append(f"Recovery ({st_from})")
        elif "PASS" in st_type:
            steps.append(f"Pass ({st_from})")
        elif "SHOT" in st_type:
            if "GOAL" in st_sub and "GOAL KICK" not in st_sub:
                steps.append(f"Goal ⚽ ({st_from})")
            else:
                steps.append(f"Shot 🎯 ({st_from})")
                
    clean_steps = []
    for step in steps:
        if not clean_steps or step != clean_steps[-1]:
            clean_steps.append(step)
            
    chain_summary = " ➔ ".join(clean_steps) if clean_steps else f"Possession ➔ {event_row['Danger_Category']} ({event_row['From']})"
    duration_sec = (seq_end_frame - seq_start_frame) / fps

    return pd.Series({
        "Seq_Start_Frame": seq_start_frame,
        "Key_Action_Frame": target_frame,
        "Seq_End_Frame": seq_end_frame,
        "Seq_Duration_s": round(duration_sec, 2),
        "Outcome_Type": outcome_type,
        "Outcome_Desc": outcome_desc,
        "Attack_Chain": chain_summary
    })

# تطبيق الدالة المحسنة
cols_to_drop = [c for c in ["Seq_Start_Frame", "Key_Action_Frame", "Seq_End_Frame", "Seq_Duration_s", "Outcome_Type", "Outcome_Desc", "Attack_Chain"] if c in df_elite.columns]
df_base = df_elite.drop(columns=cols_to_drop)

sequence_results = df_elite.apply(lambda row: trace_attack_sequence_robust(row, df_events), axis=1)
df_elite_final = pd.concat([df_base, sequence_results], axis=1)

OUTPUT_CSV = "elite_sequences_16_ready.csv"
df_elite_final.to_csv(OUTPUT_CSV, index=False)

print(f"✅ تم الحفظ بنجاح في '{OUTPUT_CSV}'!")
print("\n🎯 توزيع النتائج النهائي الدقيق:")
print(df_elite_final["Outcome_Type"].value_counts())
print("\n📋 الأهداف المؤكدة:")
print(df_elite_final[df_elite_final["Outcome_Type"] == "GOAL"][["Start Time [s]", "Team", "From", "Danger_Category", "Outcome_Desc"]].to_string(index=False))

✅ تم الحفظ بنجاح في 'elite_sequences_16_ready.csv'!

🎯 توزيع النتائج النهائي الدقيق:
Outcome_Type
SAVED          6
GOAL           3
OFF_TARGET     3
OUT            2
TURNOVER       1
PROGRESSION    1
Name: count, dtype: int64

📋 الأهداف المؤكدة:
 Start Time [s] Team     From    Danger_Category           Outcome_Desc
          91.56 Home  Player9        Goal Scored  Goal Scored (Player9)
        3600.20 Home Player10        Goal Scored Goal Scored (Player10)
        3958.32 Home  Player4 Line-Breaking Pass  Goal Scored (Player9)


In [177]:
print("الملف اتقرا؟      ", df_passes is not None)
if df_passes is not None:
    print("عدد الصفوف:       ", len(df_passes))
    print("الأعمدة:          ", df_passes.columns.tolist())
    print(df_passes.head(3).to_string())
    print("فيه Packed_Defenders؟", "Packed_Defenders" in df_passes.columns)

الملف اتقرا؟       True
عدد الصفوف:        218
الأعمدة:           ['Team', 'Type', 'Subtype', 'Period', 'Start Frame', 'Start Time [s]', 'End Frame', 'End Time [s]', 'From', 'To', 'Start X', 'Start Y', 'End X', 'End Y', 'Corridor_Packed']
   Team  Type Subtype  Period  Start Frame  Start Time [s]  End Frame  End Time [s]      From        To  Start X  Start Y  End X  End Y  Corridor_Packed
0  Away  PASS     NaN       1           45            1.80         61          2.44  Player15  Player19    57.75    55.08  47.25  46.92                1
1  Away  PASS     NaN       1          191            7.64        217          8.68  Player21  Player22    42.00    18.36  33.60   1.36                1
2  Home  PASS    HEAD       1          498           19.92        536         21.44   Player2   Player9    37.80    22.44  55.65  27.88                2
فيه Packed_Defenders؟ False


In [179]:
import json, pandas as pd
db = json.load(open("top_analyst_events.json", encoding="utf-8"))
df = pd.read_csv("elite_sequences_18.csv")
df["has_report"] = df["Key_Action_Frame"].apply(lambda f: f"frame_{f}" in db)
print(df["has_report"].value_counts())
print(df[~df["has_report"]][["Period", "Start Time [s]", "Team", "Outcome_Type", "Key_Action_Frame"]])

has_report
True    18
Name: count, dtype: int64
Empty DataFrame
Columns: [Period, Start Time [s], Team, Outcome_Type, Key_Action_Frame]
Index: []


In [180]:
import json
import pandas as pd

# 1. قراءة مفاتيح الفريمات الموجودة فعلياً في الـ JSON
with open("top_analyst_events.json", "r", encoding="utf-8") as f:
  analyst_db = json.load(f)

# استخراج أرقام الفريمات المتاحة (مثلاً: 2289 من frame_2289)
valid_frames = [
    int(k.replace("frame_", "")) for k in analyst_db.keys() if "frame_" in k
]

# 2. قراءة ملف الـ 22 هجمة وفلترته ليطابق الـ JSON 100%
df_22 = pd.read_csv("elite_sequences_18.csv")
df_matched = df_22[
    df_22["Key_Action_Frame"].isin(valid_frames)
    | df_22["Start Frame"].isin(valid_frames)
].copy()

# 3. حفظ الملف
df_matched.to_csv("elite_sequences_18.csv", index=False)
print(f"تم! دلوقتي كل اللقطات ({len(df_matched)} لقطة) ليها تقارير كاملة في الـ JSON.")

تم! دلوقتي كل اللقطات (18 لقطة) ليها تقارير كاملة في الـ JSON.


In [181]:
import json, pandas as pd
db = json.load(open("top_analyst_events.json", encoding="utf-8"))
df = pd.read_csv("elite_sequences_18.csv")

tmpl = [k for k, v in db.items()
        if str(v.get("tactical_analysis", {}).get("analysis_en", "")).startswith("Attack by")
        and "lasting" in str(v["tactical_analysis"]["analysis_en"])]
print("عدد التقارير القالب:", len(tmpl))

fr = [int(k.replace("frame_", "")) for k in tmpl]
cols = ["Key_Action_Frame", "Period", "Start Time [s]", "Team", "From", "Danger_Category",
        "Outcome_Type", "Outcome_Desc", "Attack_Chain", "Danger_Value", "Seq_Duration_s"]
print(df[df["Key_Action_Frame"].isin(fr)][cols].to_string())

عدد التقارير القالب: 0
Empty DataFrame
Columns: [Key_Action_Frame, Period, Start Time [s], Team, From, Danger_Category, Outcome_Type, Outcome_Desc, Attack_Chain, Danger_Value, Seq_Duration_s]
Index: []


In [210]:
# ==============================================================
# توليد تقارير LLM للـ 16 هجمة مباشرة مع ضبط وقت الانتظار ومعالجة الـ Rate Limit
# يعتمد على: client, SELECTED_MODEL, OUTPUT_FILE
# ==============================================================
import json
import os
import time
import pandas as pd

SEQ_FILE = "elite_sequences_16_ready.csv"
PASSES_FILE = "calculated_packing_passes.csv"

# وقت الانتظار بين كل طلب وطلب بالثواني (معدل آمن لتفادي الـ 429 Too Many Requests)
DELAY_SECONDS = 4.0
MAX_RETRIES = 3

# قراءة الملف المتفلتر الجاهز
df_seq = pd.read_csv(SEQ_FILE)
df_passes_calc = pd.read_csv(PASSES_FILE) if os.path.exists(PASSES_FILE) else None

print(f"📊 تم قراءة {len(df_seq)} هجمة جاهزة للتوليد مباشرة.")

SYSTEM_PROMPT_ANALYST = """You are an elite European football performance analyst.
Analyze the provided attacking sequence and return strictly a valid JSON object matching this schema:
{
  "analysis_ar": "تحليل تكتيكي مركز ومكثف بالعربية (3-4 أسطر)",
  "analysis_en": "Concise tactical breakdown in English (3-4 lines)",
  "defensive_structure_ar": "تقييم هيكل الخصم الدفاعي بالعربية",
  "defensive_structure_en": "Opponent defensive structure evaluation in English",
  "corridor_evaluation_ar": "تحليل كسر الخطوط والممر المستغل بالعربية",
  "corridor_evaluation_en": "Corridor and line-breaking assessment in English",
  "tactical_value_ar": "القيمة التكتيكية المضافة للهجمة بالعربية",
  "tactical_value_en": "Tactical value created in English"
}
Strict rules:
- Use ONLY the facts given (players, chain, outcome, defenders bypassed). Do not invent formations, numerical superiority or pitch zones that are not stated.
- Respect the outcome exactly: TURNOVER = possession was lost; OFF_TARGET = shot missed; SAVED = goalkeeper saved; BLOCKED = defender blocked; GOAL = goal scored.
- If defenders bypassed is 0, do not describe a line-breaking pass.
"""

# تجهيز الـ JSON وقراءة المسجل مسبقاً
analyst_top_db = {}
if os.path.exists(OUTPUT_FILE):
    try:
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            analyst_top_db = json.load(f)
            print(f"🔄 تم العثور على {len(analyst_top_db)} تقرير محفوظ مسبقاً.")
    except Exception:
        analyst_top_db = {}

# حلقة على الـ 16 صف
for n, (_, row) in enumerate(df_seq.iterrows(), 1):
    frame_id = int(row["Key_Action_Frame"])
    event_key = f"frame_{frame_id}"
    
    # تخطي اللقطات التي تم توليدها وحفظها بالفعل
    if event_key in analyst_top_db and analyst_top_db[event_key].get("tactical_analysis"):
        print(f"⏩ تخطي [{n}/{len(df_seq)}] {event_key} (تم توليده مسبقاً)")
        continue
    
    time_sec = float(row["Start Time [s]"])
    minute, second = int(time_sec // 60), int(time_sec % 60)
    team, player = str(row["Team"]), str(row["From"])
    period = int(row["Period"])
    cat = str(row["Danger_Category"])
    packed = int(row["Danger_Value"])

    # تجهيز سياق التمريرة الكاسرة للخطوط
    pass_txt = "No line-breaking pass in this sequence."
    if cat == "Line-Breaking Pass" and packed >= 3:
        receiver, coords = "Teammate", ""
        if df_passes_calc is not None:
            pm = df_passes_calc[
                (df_passes_calc["Start Frame"] == frame_id) & 
                (df_passes_calc["Period"] == period)
            ]
            if not pm.empty:
                p = pm.iloc[0]
                receiver = str(p.get("To", "Teammate"))
                if all(c in p.index for c in ["Start X", "Start Y", "End X", "End Y"]):
                    coords = (f" (from ({p['Start X']:.1f},{p['Start Y']:.1f}) to "
                              f"({p['End X']:.1f},{p['End Y']:.1f}) on a 105x68m pitch)")
        pass_txt = (f"Key line-breaking pass: {player} -> {receiver}{coords}, "
                    f"bypassing {packed} defenders inside the corridor.")

    user_content = f"""
Event: {cat}
- Time: {minute:02d}:{second:02d} (Period {period})
- Team: {team} | Key player: {player}
- Attack chain: {row['Attack_Chain']}
- Sequence duration: {row['Seq_Duration_s']} seconds
- Outcome: {row['Outcome_Type']} ({row['Outcome_Desc']})
- {pass_txt}
"""

    success = False
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=SELECTED_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT_ANALYST},
                    {"role": "user", "content": user_content},
                ],
                response_format={"type": "json_object"},
                temperature=0.2,
                max_tokens=1400,
            )
            result = json.loads(response.choices[0].message.content)

            analyst_top_db[event_key] = {
                "start_frame": frame_id,
                "period": period,
                "time_str": f"{minute:02d}:{second:02d}",
                "time_seconds": time_sec,
                "team": team,
                "player": player,
                "danger_category": cat,
                "danger_value": float(row["Danger_Value"]),
                "outcome_type": str(row["Outcome_Type"]),
                "tactical_analysis": result,
            }

            # حفظ مباشر بعد كل لقطة ناجحة
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(analyst_top_db, f, ensure_ascii=False, indent=2)

            print(f"✓ [{n}/{len(df_seq)}] {event_key} ({cat} - {minute:02d}:{second:02d} - {row['Outcome_Type']})")
            success = True
            
            # فترة الانتظار المقررة لتفادي الضغط على الـ API
            time.sleep(DELAY_SECONDS)
            break

        except Exception as e:
            err_msg = str(e)
            if "429" in err_msg or "rate" in err_msg.lower():
                wait_time = attempt * 15
                print(f"⚠️ تجاوز معدل الطلبات (Rate limit). انتظام لمدة {wait_time} ثانية (محاولة {attempt}/{MAX_RETRIES})...")
                time.sleep(wait_time)
            else:
                print(f"❌ خطأ في {event_key}: {e}")
                time.sleep(DELAY_SECONDS)
                break

    if not success:
        print(f"⛔ فشل توليد التقرير لـ {event_key} بعد {MAX_RETRIES} محاولات.")

print("=" * 60)
print(f"✅ تم الانتهاء! التقارير المكتملة في '{OUTPUT_FILE}': {len(analyst_top_db)} من أصل {len(df_seq)}")

📊 تم قراءة 16 هجمة جاهزة للتوليد مباشرة.
🔄 تم العثور على 4 تقرير محفوظ مسبقاً.
⏩ تخطي [1/16] frame_2289 (تم توليده مسبقاً)
⏩ تخطي [2/16] frame_9628 (تم توليده مسبقاً)
⏩ تخطي [3/16] frame_18252 (تم توليده مسبقاً)
✓ [4/16] frame_19470 (Big Chance Saved - 12:58 - SAVED)
⏩ تخطي [5/16] frame_26590 (تم توليده مسبقاً)
✓ [6/16] frame_45835 (Line-Breaking Pass - 30:33 - OUT)
⚠️ تجاوز معدل الطلبات (Rate limit). انتظام لمدة 15 ثانية (محاولة 1/3)...
✓ [7/16] frame_53695 (Big Chance Saved - 35:47 - SAVED)
✓ [8/16] frame_67502 (Big Chance Saved - 45:00 - SAVED)
✓ [9/16] frame_75262 (Big Chance Saved - 50:10 - SAVED)
✓ [10/16] frame_86703 (Line-Breaking Pass - 57:48 - OUT)
✓ [11/16] frame_90005 (Goal Scored - 60:00 - GOAL)
✓ [12/16] frame_98958 (Line-Breaking Pass - 65:58 - GOAL)
✓ [13/16] frame_108611 (Big Chance Saved - 72:24 - SAVED)
✓ [14/16] frame_129678 (Line-Breaking Pass - 86:27 - TURNOVER)
✓ [15/16] frame_129973 (Line-Breaking Pass - 86:38 - PROGRESSION)
✓ [16/16] frame_133892 (Line-Breaking

In [183]:
import os

key = os.environ.get("GROQ_API_KEY")
print("Key exists:", key is not None)
if key:
  print("Key starts with:", key[:7])
  print("Key length:", len(key.strip()))

Key exists: True
Key starts with: gsk_11S
Key length: 56


In [ ]:
import pandas as pd

# قراءة الملف النظيف المعتمد
df_ready = pd.read_csv("elite_sequences_16_ready.csv")

# اختيار الأعمدة التي يطلبها Streamlit كما تظهر في الصورة
required_cols = [
    "Period",
    "Start Frame",
    "Start Time [s]",
    "Team",
    "From",
    "Type",
    "Danger_Category",
    "Danger_Value",
]

# التأكد من وجود الأعمدة وحفظ الملف بالاسم المطلوب
df_top = df_ready[required_cols].copy()
df_top.to_csv("top_danger_moments.csv", index=False)

print("✅ تم إنشاء ملف 'top_danger_moments.csv' متوافقاً مع Streamlit بنسبة 100%!")
print(f"📊 عدد الهجمات: {len(df_top)}")
print("\n📋 معاينة للبيانات المصدرة:")
print(df_top.head(5).to_string(index=False))

✅ تم إنشاء ملف 'top_danger_moments.csv' متوافقاً مع Streamlit بنسبة 100%!
📊 عدد الهجمات: 16

📋 معاينة للبيانات المصدرة:
 Period  Start Frame  Start Time [s] Team     From Type    Danger_Category  Danger_Value
      1         2289           91.56 Home  Player9 SHOT        Goal Scored          10.0
      1         9628          385.12 Home  Player8 SHOT   Big Chance Saved           8.0
      1        18252          730.08 Home  Player3 PASS Line-Breaking Pass           3.0
      1        19470          778.80 Away Player24 SHOT   Big Chance Saved           8.0
      1        26590         1063.60 Home Player10 PASS Line-Breaking Pass           4.0


: 

=======================================================================================================================================================================================================================

Casual

In [2]:
# ==============================================================
# الخطوة 1: استخراج صفوة الـ 11 لقطة المعتمدة للمشجع العادي
# الملف الناتج: casual_elite_11_moments.csv
# ==============================================================
import pandas as pd

SEQ_FILE = "elite_sequences_16_ready.csv"
OUTPUT_FILE = "casual_elite_11_moments.csv"

df_seq = pd.read_csv(SEQ_FILE)

# 1. الأهداف الثلاثة الصريحة
goals = df_seq[df_seq["Outcome_Type"] == "GOAL"].copy()
goals["Category_Key"] = "goals"
goals["Category_Name"] = "الأهداف ولحظات الحسم ⚽"

# 2. أفضل 4 تصديات إعجازية (مرتبة حسب زمن الهجمة وسرعة ارتدادها)
saves = (
    df_seq[
        (df_seq["Outcome_Type"] == "SAVED")
        | (df_seq["Danger_Category"] == "Big Chance Saved")
    ]
    .sort_values(by="Danger_Value", ascending=False)
    .head(4)
    .copy()
)
saves["Category_Key"] = "saves"
saves["Category_Name"] = "تصديات إعجازية وقفاز ذهبي 🧤"

# 3. أفضل 4 تمريرات ليزرية (نبدأ بالأعلى في Danger_Value: الـ 5 والـ 4 والـ 3)
passes = (
    df_seq[
        (df_seq["Danger_Category"] == "Line-Breaking Pass")
        & (df_seq["Outcome_Type"] != "GOAL")
    ]
    .sort_values(by="Danger_Value", ascending=False)
    .head(4)
    .copy()
)
passes["Category_Key"] = "passes"
passes["Category_Name"] = "تمريرات ليزرية كاسرة للخطوط 🎯"

# دمج النخبة (3 أهداف + 4 تصديات + 4 تمريرات = 11 لقطة أيقونية تماماً)
df_curated = pd.concat([goals, saves, passes], ignore_index=True)

# حفظ الملف
df_curated.to_csv(OUTPUT_FILE, index=False)

print(f"✅ تم الحفظ بنجاح في: '{OUTPUT_FILE}'")
print(f"📊 التوزيع النهائي:")
print(f"   - أهداف: {len(goals)} لقطات")
print(f"   - تصديات: {len(saves)} لقطات")
print(f"   - تمريرات ليزرية: {len(passes)} لقطات")
print(f"📌 الإجمالي: {len(df_curated)} لقطة جاهزة لتوليد تعليق الـ LLM!")

✅ تم الحفظ بنجاح في: 'casual_elite_11_moments.csv'
📊 التوزيع النهائي:
   - أهداف: 3 لقطات
   - تصديات: 4 لقطات
   - تمريرات ليزرية: 4 لقطات
📌 الإجمالي: 11 لقطة جاهزة لتوليد تعليق الـ LLM!


In [10]:
# ==============================================================
# الخطوة 2 (المعدلة): توليد تعليق متزامن عربي وإنجليزي مع تفادي KeyError
# الملف الناتج: casual_fan_moments.json
# ==============================================================
import json
import os
import re
import time
from groq import Groq, RateLimitError
import pandas as pd

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

PRIMARY_MODEL = "openai/gpt-oss-120b"
FALLBACK_MODEL = "allam-2-7b"

INPUT_CSV = "casual_elite_11_moments.csv"
OUTPUT_JSON = "casual_fan_moments.json"

df_curated = pd.read_csv(INPUT_CSV)

# تنظيف أسماء الأعمدة من المسافات وعلامة '-' التي كانت في '-Period'
df_curated.columns = df_curated.columns.str.strip().str.lstrip("-")


def build_bilingual_prompt(row):
    return f"""
You are a professional TV football play-by-play commentator and match analyst.
Generate dynamic, synchronized play-by-play commentary in both Arabic and English based on the tracked ball movement chain.
Do not reveal the final outcome until the last climax moment.

Match Event Details:
- Team: {row.get('Team', 'Home')}
- Key Player: {row.get('From', 'Player')}
- Category: {row.get('Category_Name', '')}
- Total Sequence Duration: {row.get('Seq_Duration_s', 6.0)} seconds
- Ball Progression Chain: {row.get('Attack_Chain', '')}
- Final Outcome: {row.get('Outcome_Type', '')} ({row.get('Outcome_Desc', '')})
- Danger / Broken Defenders: {row.get('Danger_Value', 0)}

Requirements:
Break down the sequence into 3 synchronized phases matching the ball movement:
1. "build_up" (t=0.0s): Ball recovery or origin of the attack.
2. "mid_action" (midpoint): The decisive through pass, progression, and breaking defensive lines.
3. "climax" (final second): The strike, save, or goal moment.

Provide the commentary in natural, exciting Arabic (العامية الرياضية الراقية) and its direct English broadcast translation.

Return ONLY a valid JSON object strictly following this structure:
{{
  "card_title_ar": "عنوان مشوق ومختصر للقطة كأنه مانشيت تلفزيوني",
  "card_title_en": "Catchy broadcast headline for the moment",
  "card_badge_ar": "شارة مميزة (مثل: هدف ملعوب ⚽ أو قفاز الأخطبوط 🧤)",
  "card_badge_en": "Short action badge (e.g. Clinical Goal ⚽ or Epic Save 🧤)",
  "timeline": [
    {{
      "stage": "build_up",
      "time_offset_s": 0.0,
      "text_ar": "تعليق البداية عند استلام الكرة وبناء الهجمة بالعربي",
      "text_en": "Opening commentary as the attack initiates in English"
    }},
    {{
      "stage": "mid_action",
      "time_offset_s": 2.5,
      "text_ar": "تعليق نقل الكرة وكسر المدافعين بالعربي",
      "text_en": "Mid-action commentary covering the key pass and line-breaking move in English"
    }},
    {{
      "stage": "climax",
      "time_offset_s": 5.5,
      "text_ar": "صراخ وحماس المعلق في لحظة الشوطة أو الهدف أو التصدي بالعربي",
      "text_en": "High-energy reaction at the final strike, save, or goal in English"
    }}
  ]
}}
"""


def call_groq_api(prompt):
    models = [PRIMARY_MODEL, FALLBACK_MODEL, "openai/gpt-oss-20b"]
    for model_name in models:
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {
                            "role": "system",
                            "content": (
                                "You are a football broadcast commentator."
                                " Return ONLY valid JSON matching the exact"
                                " requested schema."
                            ),
                        },
                        {"role": "user", "content": prompt},
                    ],
                    response_format={"type": "json_object"},
                    temperature=0.7,
                )
                return response.choices[0].message.content
            except RateLimitError:
                wait_sec = 12 * (attempt + 1)
                print(
                    f"⏳ ضغط طلبات على {model_name}.. جاري الانتظار"
                    f" {wait_sec} ثانية..."
                )
                time.sleep(wait_sec)
            except Exception as e:
                print(f"⚠️ خطأ مع {model_name}: {e}")
                time.sleep(3)
                break
    return None


print(
    "🚀 جاري معالجة الـ 11 لقطة وتوليد التعليق اللحظي (عربي/إنجليزي) عبر"
    " Groq..."
)

casual_data = {"categories": {"goals": [], "saves": [], "passes": []}}

for idx, row in df_curated.iterrows():
    frame_id = int(row["Key_Action_Frame"])
    cat_key = str(row.get("Category_Key", "goals"))
    time_sec = float(row.get("Start Time [s]", 0.0))
    minute, second = int(time_sec // 60), int(time_sec % 60)
    dur = float(row.get("Seq_Duration_s", 6.0))

    prompt = build_bilingual_prompt(row)
    raw_response = call_groq_api(prompt)

    try:
        parsed_res = json.loads(raw_response)
    except Exception:
        parsed_res = {
            "card_title_ar": f"لقطة خطيرة - {row.get('From', '')}",
            "card_title_en": f"Dangerous Action - {row.get('From', '')}",
            "card_badge_ar": str(row.get("Category_Name", "لقطة مميزة 🔥")),
            "card_badge_en": cat_key.capitalize(),
            "timeline": [
                {
                    "stage": "build_up",
                    "time_offset_s": 0.0,
                    "text_ar": f"بداية بناء الهجمة من {row.get('From', '')}..",
                    "text_en": (
                        f"Attack initiated through {row.get('From', '')} in"
                        " midfield.."
                    ),
                },
                {
                    "stage": "climax",
                    "time_offset_s": dur,
                    "text_ar": "إنهاء سريع ومباشر للهجمة وسط رقابة دفاعية!",
                    "text_en": (
                        "Decisive finish to the sequence under defensive"
                        " pressure!"
                    ),
                },
            ],
        }

    timeline = parsed_res.get("timeline", [])
    if len(timeline) >= 3:
        timeline[0]["time_offset_s"] = 0.0
        timeline[1]["time_offset_s"] = round(dur * 0.45, 1)
        timeline[2]["time_offset_s"] = round(dur, 1)

    item = {
        "event_id": f"frame_{frame_id}",
        "frame_id": frame_id,
        "period": int(row.get("Period", 1)),
        "time_str": f"{minute:02d}:{second:02d}",
        "start_time_s": time_sec,
        "duration_s": dur,
        "team": str(row.get("Team", "Home")),
        "key_player": str(row.get("From", "Player")),
        "card_badge_ar": parsed_res.get("card_badge_ar", "لقطة مميزة 🔥"),
        "card_badge_en": parsed_res.get("card_badge_en", "Highlight 🔥"),
        "card_title_ar": parsed_res.get(
            "card_title_ar", f"هجمة خطيرة - {row.get('From', '')}"
        ),
        "card_title_en": parsed_res.get(
            "card_title_en", f"Key Sequence - {row.get('From', '')}"
        ),
        "category": cat_key,
        "attack_chain": str(row.get("Attack_Chain", "")),
        "outcome": str(row.get("Outcome_Type", "")),
        "danger_value": float(row.get("Danger_Value", 0.0)),
        "commentator_timeline": timeline,
    }

    casual_data["categories"][cat_key].append(item)
    print(
        f"✅ [{idx+1}/11] تم تجهيز اللقطة ({item['time_str']}):"
        f" {item['card_title_ar']} | {item['card_title_en']}"
    )
    time.sleep(1.5)

# حفظ ملف الـ JSON النهائي
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(casual_data, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 55)
print(f"🎉 تم بنجاح إنشاء ملف الـ JSON المزدوج: '{OUTPUT_JSON}'")
print(
    f"📊 الإجمالي: 3 أهداف | 4 تصديات | 4 تمريرات (جميعها جاهزة بالعربي"
    f" والإنجليزي)"
)

🚀 جاري معالجة الـ 11 لقطة وتوليد التعليق اللحظي (عربي/إنجليزي) عبر Groq...
✅ [1/11] تم تجهيز اللقطة (01:31): تمرير سحري يفتح باب الهدف! | Magic Pass Opens the Door to Goal!
✅ [2/11] تم تجهيز اللقطة (60:00): لحظة حاسمة: هدف اللاعب 10 بعد سلسلة سريعة | Decisive Moment: Player 10’s Lightning Goal
✅ [3/11] تم تجهيز اللقطة (65:58): هجمة صاعقة تنتهي بشبكة صافية | Lightning Counter Ends with a Crisp Finish
✅ [4/11] تم تجهيز اللقطة (06:25): تصدي إعجازي من قفاز ذهبي للـحارس! | Miraculous Golden Glove Save Stuns the Crowd!
✅ [5/11] تم تجهيز اللقطة (12:58): تصدي إعجازي للكرة الذهبية | Golden Glove Miracle Save
✅ [6/11] تم تجهيز اللقطة (35:47): قفاز ذهبي يلمع في تصدي اللاعب 8! | Golden Glove shines as Player8 makes a miraculous save!
✅ [7/11] تم تجهيز اللقطة (45:00): تصدي إعجازي بقبضة ذهبية من اللاعب 9 | Miraculous Save by Player 9's Golden Glove
✅ [8/11] تم تجهيز اللقطة (17:43): تمرير ليزري يفتح الدفاع! | Laser Pass Breaks the Lines!
✅ [9/11] تم تجهيز اللقطة (12:10): تمريرات ليزرية تكسر الخطوط! |

========================================================================================================================================================================================================================

Player Spotlight

In [5]:
# ==============================================================================
# المرحلة الأولى: استخراج المراكز وتوحيد الأحداث آلياً بالكامل (Zero Hardcoding)
# المخرجات: all_players_master_events.csv
# ==============================================================================
import os
import pandas as pd
import numpy as np

EVENTS_FILE = "Sample_Game_1_RawEventsData.csv" if os.path.exists("Sample_Game_1_RawEventsData.csv") else "events.csv"
HOME_FILE = "Sample_Game_1_RawTrackingData_Home_Team.csv"
AWAY_FILE = "Sample_Game_1_RawTrackingData_Away_Team.csv"
PASSES_FILE = "calculated_packing_passes.csv"
OUTPUT_MASTER_CSV = "all_players_master_events.csv"

FPS = 25
PITCH_L, PITCH_W = 105.0, 68.0

print(f"⏳ جاري قراءة ملف الأحداث والتتبع...")
df_events = pd.read_csv(EVENTS_FILE)
df_events.columns = [str(c).strip() for c in df_events.columns]

df_passes = None
if os.path.exists(PASSES_FILE):
    df_passes = pd.read_csv(PASSES_FILE)
    df_passes.columns = [str(c).strip() for c in df_passes.columns]

# 1. قراءة التراكينج لتحديد المراكز هندسياً
def load_metrica_tracking(csv_path, team_prefix):
    with open(csv_path, 'r', encoding='utf-8') as f:
        _ = f.readline()
        row1 = f.readline().strip().split(',')
        _ = f.readline()
    
    col_names = ['Period', 'Frame', 'Time [s]']
    i = 3
    while i < len(row1):
        if i >= len(row1) - 2:
            col_names.extend(['Ball_x', 'Ball_y'])
            break
        num = str(row1[i]).strip()
        if not num: num = str(row1[i-1]).strip()
        col_names.append(f"{team_prefix}_Player{num}_x")
        col_names.append(f"{team_prefix}_Player{num}_y")
        i += 2
        
    df = pd.read_csv(csv_path, skiprows=3, header=None, names=col_names, low_memory=False)
    for c in df.columns:
        if c not in ['Period', 'Frame']:
            df[c] = pd.to_numeric(df[c], errors='coerce')
            
    x_cols = [c for c in df.columns if c.endswith('_x')]
    y_cols = [c for c in df.columns if c.endswith('_y')]
    if x_cols and np.nanmax(df[x_cols].to_numpy(dtype=float)) <= 1.5:
        for c in x_cols: df[c] = df[c] * PITCH_L
        for c in y_cols: df[c] = (1.0 - df[c]) * PITCH_W
    return df

df_home = load_metrica_tracking(HOME_FILE, "Home")
df_away = load_metrica_tracking(AWAY_FILE, "Away")

# 2. تحديد الحراس ديناميكياً لكل شوط (اللاعب الأكثر عمقاً قرب مرماه)
def detect_goalkeepers_dynamically(df_team, prefix):
    gks_by_period = {}
    p_cols_x = [c for c in df_team.columns if c.startswith(f"{prefix}_Player") and c.endswith('_x')]
    
    for period in [1, 2]:
        p_sub = df_team[df_team['Period'] == period]
        if p_sub.empty:
            continue
        
        # معرفة اتجاه اللعب (هل المرمى عند X=0 أم X=105)
        mean_all_x = np.nanmean(p_sub[p_cols_x].to_numpy())
        goal_x = 0.0 if mean_all_x < 52.5 else 105.0
        
        # حساب المسافة لخط المرمى لكل لاعب في هذا الشوط
        player_distances = {}
        for cx in p_cols_x:
            pid = cx.replace(f"{prefix}_Player", "").replace("_x", "")
            valid_x = p_sub[cx].dropna()
            if len(valid_x) > 500: # شارك في عدد فريمات كافٍ
                dist_to_goal = np.abs(valid_x - goal_x).mean()
                player_distances[pid] = dist_to_goal
                
        if player_distances:
            # اللاعب الأقرب بوضوح لخط مرماه هو الحارس لهذا الشوط
            best_gk = min(player_distances, key=player_distances.get)
            gks_by_period[period] = best_gk
            
    return gks_by_period

home_gks = detect_goalkeepers_dynamically(df_home, "Home")
away_gks = detect_goalkeepers_dynamically(df_away, "Away")

print(f"🧤 الحراس المكتشفين تلقائياً:")
print(f"   Home: الشوط الأول={home_gks.get(1)}, الشوط الثاني={home_gks.get(2)}")
print(f"   Away: الشوط الأول={away_gks.get(1)}, الشوط الثاني={away_gks.get(2)}")

# 3. استخراج الأحداث بدون أي Hardcoding
def clean_player_id(raw_val):
    if pd.isna(raw_val): return None
    return str(raw_val).strip().replace("Player", "").replace("player", "").strip()

def format_time_seconds(time_val):
    t_sec = float(time_val)
    return f"{int(t_sec // 60):02d}:{int(t_sec % 60):02d}"

master_records = []

for idx, row in df_events.iterrows():
    team = str(row.get("Team", "")).strip()
    event_type = str(row.get("Type", "")).strip().upper()
    subtype = str(row.get("Subtype", "")).strip().upper()
    from_player = clean_player_id(row.get("From"))
    to_player = clean_player_id(row.get("To"))
    
    s_frame = int(row["Start Frame"])
    e_frame = int(row["End Frame"]) if pd.notna(row.get("End Frame")) and int(row["End Frame"]) > s_frame else s_frame + 75
    period = int(row.get("Period", 1))
    time_s = float(row.get("Start Time [s]", s_frame / FPS))
    time_str = format_time_seconds(time_s)
    
    start_x = float(row.get("Start X", 0.0)) * PITCH_L if pd.notna(row.get("Start X")) else np.nan
    start_y = float(row.get("Start Y", 0.0)) * PITCH_W if pd.notna(row.get("Start Y")) else np.nan

    # أ) التسديدات
    if "SHOT" in event_type and from_player:
        outcome = "OFF_TARGET"
        metric_score = 1
        if "GOAL" in subtype: outcome, metric_score = "GOAL", 5
        elif "SAVED" in subtype or "ON TARGET" in subtype: outcome, metric_score = "ON_TARGET", 3
        elif "BLOCKED" in subtype: outcome, metric_score = "BLOCKED", 2

        master_records.append({
            "player_id": from_player, "team": team, "action_category": "ATTACK",
            "event_type": "SHOT", "subtype": subtype or "SHOT", "outcome": outcome,
            "metric_value": metric_score, "period": period, "start_frame": max(1, s_frame - 75),
            "end_frame": e_frame + 75, "key_frame": s_frame, "time_seconds": round(time_s, 2),
            "time_str": time_str, "pos_x": round(start_x, 1) if pd.notna(start_x) else None,
            "pos_y": round(start_y, 1) if pd.notna(start_y) else None,
            "raw_note": f"تسديدة من اللاعب ({outcome})"
        })

    # ب) تصديات الحراس التلقائية
    if "SHOT" in event_type and "SAVED" in subtype:
        defending_team = "Away" if team == "Home" else "Home"
        
        # اختيار الحارس: إذا وجد في To، وإلا نأخذ الحارس المكتشف آلياً لهذا الشوط
        if to_player:
            actual_gk = to_player
        else:
            actual_gk = home_gks.get(period) if defending_team == "Home" else away_gks.get(period)

        master_records.append({
            "player_id": actual_gk, "team": defending_team, "action_category": "GOALKEEPER",
            "event_type": "SAVE", "subtype": "GOAL_SAVE", "outcome": "SAVED",
            "metric_value": 4, "period": period, "start_frame": max(1, s_frame - 75),
            "end_frame": e_frame + 75, "key_frame": s_frame, "time_seconds": round(time_s, 2),
            "time_str": time_str, "pos_x": round(start_x, 1) if pd.notna(start_x) else None,
            "pos_y": round(start_y, 1) if pd.notna(start_y) else None,
            "raw_note": f"تصدٍّ لمرمى {defending_team}"
        })

    # ج) التدخلات الدفاعية العادية (مع استثناء الحراس آلياً)
    if any(k in event_type or k in subtype for k in ["BALL OUT", "RECOVERY", "TACKLE", "INTERCEPTION"]):
        if from_player:
            # التحقق هل هذا اللاعب هو حارس فريقه في هذا الشوط؟
            current_team_gks = home_gks if team == "Home" else away_gks
            is_active_gk = (from_player == current_team_gks.get(period))

            master_records.append({
                "player_id": from_player, "team": team,
                "action_category": "GOALKEEPER" if is_active_gk else "DEFENSE",
                "event_type": "GK_ACTION" if is_active_gk else "DEFENSIVE_ACTION",
                "subtype": subtype or event_type, "outcome": "SUCCESS",
                "metric_value": 2, "period": period, "start_frame": max(1, s_frame - 60),
                "end_frame": e_frame + 60, "key_frame": s_frame, "time_seconds": round(time_s, 2),
                "time_str": time_str, "pos_x": round(start_x, 1) if pd.notna(start_x) else None,
                "pos_y": round(start_y, 1) if pd.notna(start_y) else None,
                "raw_note": f"تدخل ناجح ({subtype or event_type})"
            })

# 4. التمريرات الكاسرة للخطوط
if df_passes is not None and not df_passes.empty:
    passer_cols = [c for c in df_passes.columns if "from" in c.lower() or "passer" in c.lower()]
    packed_cols = [c for c in df_passes.columns if "packed" in c.lower() or "danger" in c.lower()]
    receiver_cols = [c for c in df_passes.columns if "to" in c.lower() or "receiver" in c.lower()]
    
    p_col = passer_cols[0] if passer_cols else None
    pk_col = packed_cols[0] if packed_cols else None
    r_col = receiver_cols[0] if receiver_cols else None

    if p_col and pk_col:
        for idx, row in df_passes.iterrows():
            passer_id = clean_player_id(row.get(p_col))
            receiver_id = clean_player_id(row.get(r_col)) if r_col else None
            packed_val = int(row.get(pk_col, 0))
            team = str(row.get("Team", "")).strip()

            s_frame = int(row["Start Frame"])
            e_frame = int(row["End Frame"]) if "End Frame" in row and pd.notna(row["End Frame"]) and int(row["End Frame"]) > s_frame else s_frame + 60
            period = int(row.get("Period", 1))
            time_s = float(row.get("Time [s]", s_frame / FPS))
            time_str = format_time_seconds(time_s)

            if passer_id and packed_val > 0:
                master_records.append({
                    "player_id": passer_id, "team": team, "action_category": "PLAYMAKING",
                    "event_type": "PACKING_PASS", "subtype": f"PACKED_{packed_val}",
                    "outcome": "PASS_KEY", "metric_value": packed_val,
                    "period": period, "start_frame": max(1, s_frame - 60),
                    "end_frame": e_frame + 60, "key_frame": s_frame, "time_seconds": round(time_s, 2),
                    "time_str": time_str, "pos_x": None, "pos_y": None,
                    "raw_note": f"تمريرة كاسرة لـ {packed_val} مدافعين"
                })

            if receiver_id and packed_val >= 2:
                master_records.append({
                    "player_id": receiver_id, "team": team, "action_category": "POSITIONING",
                    "event_type": "KEY_RECEPTION", "subtype": f"RECEIVED_BEHIND_{packed_val}",
                    "outcome": "PACKING_RECEIVE", "metric_value": packed_val,
                    "period": period, "start_frame": max(1, s_frame - 60),
                    "end_frame": e_frame + 60, "key_frame": s_frame, "time_seconds": round(time_s, 2),
                    "time_str": time_str, "pos_x": None, "pos_y": None,
                    "raw_note": "استلام تمريرة بين الخطوط"
                })

df_master = pd.DataFrame(master_records)
df_master = df_master.dropna(subset=["player_id"]).drop_duplicates(subset=["player_id", "key_frame", "event_type"])
df_master.sort_values(by=["team", "player_id", "time_seconds"], inplace=True)
df_master.to_csv(OUTPUT_MASTER_CSV, index=False, encoding="utf-8")

print("=" * 60)
print(f"🎉 تم استخراج '{OUTPUT_MASTER_CSV}' بمنطق رياضي سليم بالكامل.")
print(f"📊 إجمالي الأحداث: {len(df_master)}")

⏳ جاري قراءة ملف الأحداث والتتبع...
🧤 الحراس المكتشفين تلقائياً:
   Home: الشوط الأول=11, الشوط الثاني=11
   Away: الشوط الأول=25, الشوط الثاني=25
🎉 تم استخراج 'all_players_master_events.csv' بمنطق رياضي سليم بالكامل.
📊 إجمالي الأحداث: 818


In [12]:
# ==============================================================================
# استخراج بروفايل شامل مع حساب دقة التمرير الحقيقية والمراكز ديناميكياً
# المخرجات: players_unfiltered_raw_profiles.json
# ==============================================================================
import json
import os
import numpy as np
import pandas as pd

MASTER_CSV = "all_players_master_events.csv"
HOME_FILE = "Sample_Game_1_RawTrackingData_Home_Team.csv"
AWAY_FILE = "Sample_Game_1_RawTrackingData_Away_Team.csv"
EVENTS_FILE = "Sample_Game_1_RawEventsData.csv" if os.path.exists("Sample_Game_1_RawEventsData.csv") else "events.csv"
OUTPUT_UNFILTERED_JSON = "players_unfiltered_raw_profiles.json"

print("⏳ جاري قراءة البيانات...")
df_master = pd.read_csv(MASTER_CSV)
df_events = pd.read_csv(EVENTS_FILE)
df_events.columns = [str(c).strip() for c in df_events.columns]

# 1. قراءة التراكينج
def load_metrica_tracking(csv_path, team_prefix):
    with open(csv_path, 'r', encoding='utf-8') as f:
        _ = f.readline()
        row1 = f.readline().strip().split(',')
        _ = f.readline()
    
    col_names = ['Period', 'Frame', 'Time [s]']
    i = 3
    while i < len(row1):
        if i >= len(row1) - 2:
            col_names.extend(['Ball_x', 'Ball_y'])
            break
        num = str(row1[i]).strip()
        if not num: num = str(row1[i-1]).strip()
        col_names.append(f"{team_prefix}_Player{num}_x")
        col_names.append(f"{team_prefix}_Player{num}_y")
        i += 2
        
    df = pd.read_csv(csv_path, skiprows=3, header=None, names=col_names, low_memory=False)
    for c in df.columns:
        if c not in ['Period', 'Frame']:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

df_home = load_metrica_tracking(HOME_FILE, "Home")
df_away = load_metrica_tracking(AWAY_FILE, "Away")

# 2. تحديد المراكز هندسياً
def determine_team_roles(df_team, prefix, detected_gk_id):
    p_cols = [c for c in df_team.columns if c.startswith(f"{prefix}_Player") and c.endswith('_x')]
    p1 = df_team[df_team['Period'] == 1]
    all_x = np.nanmean(p1[p_cols].to_numpy())
    own_goal_x = 0.0 if all_x < 0.5 else 1.0
    
    player_avg_dist = {}
    for c in p_cols:
        pid = c.replace(f"{prefix}_Player", "").replace("_x", "")
        valid = df_team[c].dropna()
        if len(valid) > 500:
            dist = np.abs(valid - own_goal_x).mean()
            player_avg_dist[pid] = dist
            
    field_players = {pid: d for pid, d in player_avg_dist.items() if pid != str(detected_gk_id)}
    sorted_field = sorted(field_players.items(), key=lambda x: x[1])
    total_field = len(sorted_field)
    
    roles = {str(detected_gk_id): "Goalkeeper"}
    if total_field > 0:
        n_def = max(1, int(round(total_field * 0.38)))
        n_fwd = max(1, int(round(total_field * 0.22)))
        for i, (pid, _) in enumerate(sorted_field):
            if i < n_def:
                roles[pid] = "Defender"
            elif i >= total_field - n_fwd:
                roles[pid] = "Forward"
            else:
                roles[pid] = "Midfielder"
    return roles

home_roles = determine_team_roles(df_home, "Home", "11")
away_roles = determine_team_roles(df_away, "Away", "25")

# 3. دالة حساب دقة التمرير الحقيقية من تسلسل أحداث اللعب
def calculate_real_pass_accuracy(df_ev, pid_str, team_str):
    p_name = f"Player{pid_str}"
    pass_rows = df_ev[(df_ev["Team"].astype(str) == team_str) & 
                      (df_ev["From"].astype(str) == p_name) & 
                      (df_ev["Type"].str.upper().str.contains("PASS", na=False))]
    
    tot = len(pass_rows)
    if tot == 0:
        return 0, 0, 0.0
    
    comp = 0
    for p_idx in pass_rows.index:
        cur_row = df_ev.loc[p_idx]
        if p_idx + 1 < len(df_ev):
            next_row = df_ev.loc[p_idx + 1]
            n_team = str(next_row.get("Team", "")).strip()
            n_type = str(next_row.get("Type", "")).upper()
            n_sub = str(next_row.get("Subtype", "")).upper()
            
            # إذا الكرة استمرت مع نفس الفريق
            if n_team == team_str and "FAULT" not in n_type:
                comp += 1
            # إذا استخلصها الخصم
            elif n_team != team_str and any(k in n_type or k in n_sub for k in ["INTERCEPTION", "BALL LOST", "OUT"]):
                pass
            else:
                if pd.notna(cur_row.get("To")):
                    comp += 1
        else:
            if pd.notna(cur_row.get("To")):
                comp += 1
                
    acc_val = round((comp / tot) * 100, 1)
    return tot, comp, acc_val

# 4. بناء البروفايل الشامل
unfiltered_db = {}
grouped = df_master.groupby(["team", "player_id"])

for (team, pid), p_events in grouped:
    pid_str = str(pid).strip()
    roles_dict = home_roles if team == "Home" else away_roles
    role = roles_dict.get(pid_str, "Midfielder")
    p_name = f"Player{pid_str}"

    # حساب التمريرات الحقيقي بالدالة الجديدة
    tot_passes, completed_p, acc = calculate_real_pass_accuracy(df_events, pid_str, team)

    # حساب التسديدات
    team_events = df_events[df_events["Team"].astype(str) == team]
    player_from = team_events[team_events["From"].astype(str) == p_name]
    shots = player_from[player_from["Type"].str.upper().str.contains("SHOT", na=False)]
    tot_shots = len(shots)
    goals = len(shots[shots["Subtype"].astype(str).str.upper().str.contains("GOAL")])
    on_target = len(shots[shots["Subtype"].astype(str).str.upper().str.contains("GOAL|SAVED")])
    off_target = tot_shots - on_target

    # التمريرات الكاسرة للخطوط
    packing_df = p_events[p_events["action_category"] == "PLAYMAKING"]
    tot_packing = len(packing_df)
    packed_sum = int(packing_df["metric_value"].sum()) if not packing_df.empty else 0

    # التصديات للحراس
    gk_saves_df = p_events[(p_events["action_category"] == "GOALKEEPER") & (p_events["event_type"] == "SAVE")]
    tot_saves = len(gk_saves_df)

    # التدخلات الدفاعية
    def_df = p_events[p_events["action_category"] == "DEFENSE"]
    tot_def = len(def_df)

    def extract_moments(df_sub, cat_title):
        res = []
        for _, row in df_sub.sort_values(by="time_seconds").iterrows():
            t_str = str(row["time_str"])
            res.append({
                "event_id": f"evt_{pid_str}_{int(row['key_frame'])}",
                "category": str(row["action_category"]),
                "event_type": str(row["event_type"]),
                "period": int(row["period"]),
                "start_frame": int(row["start_frame"]),
                "end_frame": int(row["end_frame"]),
                "key_frame": int(row["key_frame"]),
                "time_seconds": float(row["time_seconds"]),
                "time_str": t_str,
                "metric_value": int(row["metric_value"]),
                "outcome": str(row["outcome"]),
                "title": f"{cat_title} [{t_str}]",
                "description": str(row["raw_note"])
            })
        return res

    raw_keys = p_events[p_events["action_category"].isin(["ATTACK", "PLAYMAKING", "GOALKEEPER"])]
    raw_pos = p_events[p_events["action_category"] == "POSITIONING"]
    raw_def = p_events[p_events["action_category"] == "DEFENSE"]

    player_key = f"{team}_Player{pid_str}"
    unfiltered_db[player_key] = {
        "player_id": pid_str,
        "team": team,
        "role": role,
        "full_match_stats": {
            "total_passes": tot_passes,
            "completed_passes": completed_p,
            "pass_accuracy_pct": acc,
            "total_shots": tot_shots,
            "shots_on_target": on_target,
            "shots_off_target": off_target,
            "goals": goals,
            "total_packing_passes": tot_packing,
            "total_defenders_packed": packed_sum,
            "total_saves": tot_saves,
            "total_defensive_actions": tot_def
        },
        "all_raw_moments": {
            "key_actions": extract_moments(raw_keys, "أهم لقطات اللاعب"),
            "positioning_receptions": extract_moments(raw_pos, "لقطة تمركز واستلام"),
            "defensive_actions": extract_moments(raw_def, "موقف دفاعي للاعب")
        }
    }

with open(OUTPUT_UNFILTERED_JSON, "w", encoding="utf-8") as f:
    json.dump(unfiltered_db, f, ensure_ascii=False, indent=2)

print("=" * 60)
print(f"🎉 تم تحديث: '{OUTPUT_UNFILTERED_JSON}' بنسب تمرير واقعية ومحسوبة صح!")

⏳ جاري قراءة البيانات...
🎉 تم تحديث: 'players_unfiltered_raw_profiles.json' بنسب تمرير واقعية ومحسوبة صح!


In [13]:
# ==============================================================================
# فلترة أقوى اللقطات مع تثبيت كامل الإحصائيات للكارت
# المدخلات: players_unfiltered_raw_profiles.json
# المخرجات: players_spotlight_profiles.json
# ==============================================================================
import json
import os

INPUT_FILE = "players_unfiltered_raw_profiles.json"
OUTPUT_FILE = "players_spotlight_profiles.json"

if not os.path.exists(INPUT_FILE):
  raise FileNotFoundError(f"الملف {INPUT_FILE} غير موجود.")

print(f"⏳ جاري قراءة الملف الشامل: {INPUT_FILE}")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
  raw_db = json.load(f)

spotlight_db = {}
TOP_N = 3

for player_key, p_data in raw_db.items():
  pid = p_data["player_id"]
  team = p_data["team"]
  role = p_data["role"]
  stats = p_data["full_match_stats"]
  raw_moments = p_data.get("all_raw_moments", {})

  # 1. فلترة أقوى اللقطات الهجومية وصناعة اللعب / التصديات
  keys = raw_moments.get("key_actions", [])
  sorted_keys = sorted(
      keys, key=lambda x: (x["metric_value"], -x["time_seconds"]), reverse=True
  )[:TOP_N]

  filtered_keys = []
  for m in sorted_keys:
    t_str = m["time_str"]
    filtered_keys.append({
        "event_id": m["event_id"],
        "period": m["period"],
        "start_frame": m["start_frame"],
        "end_frame": m["end_frame"],
        "key_frame": m["key_frame"],
        "time_str": t_str,
        "danger_value": m["metric_value"],
        "outcome": m["outcome"],
        "title_ar": f"أهم لقطات اللاعب [{t_str}]",
        "title_en": f"Key Player Highlight [{t_str}]",
        "desc_ar": m["description"],
        "desc_en": f"Match highlight at minute {t_str}.",
    })

  # 2. فلترة لقطات التمركز والاستلام بين الخطوط
  pos = raw_moments.get("positioning_receptions", [])
  sorted_pos = sorted(
      pos, key=lambda x: (x["metric_value"], -x["time_seconds"]), reverse=True
  )[:TOP_N]

  filtered_pos = []
  for m in sorted_pos:
    t_str = m["time_str"]
    filtered_pos.append({
        "event_id": m["event_id"],
        "period": m["period"],
        "start_frame": m["start_frame"],
        "end_frame": m["end_frame"],
        "key_frame": m["key_frame"],
        "time_str": t_str,
        "danger_value": m["metric_value"],
        "outcome": m["outcome"],
        "title_ar": f"أهم لقطات التمركز والاستلام [{t_str}]",
        "title_en": f"Key Space Reception [{t_str}]",
        "desc_ar": m["description"],
        "desc_en": f"Reception between lines at minute {t_str}.",
    })

  # 3. فلترة المواقف الدفاعية
  defense = raw_moments.get("defensive_actions", [])
  sorted_def = sorted(
      defense,
      key=lambda x: (x["metric_value"], -x["time_seconds"]),
      reverse=True,
  )[:TOP_N]

  filtered_def = []
  for m in sorted_def:
    t_str = m["time_str"]
    filtered_def.append({
        "event_id": m["event_id"],
        "period": m["period"],
        "start_frame": m["start_frame"],
        "end_frame": m["end_frame"],
        "key_frame": m["key_frame"],
        "time_str": t_str,
        "danger_value": m["metric_value"],
        "outcome": m["outcome"],
        "title_ar": f"المواقف الدفاعية للاعب [{t_str}]",
        "title_en": f"Defensive Play [{t_str}]",
        "desc_ar": m["description"],
        "desc_en": f"Defensive involvement at minute {t_str}.",
    })

  # بناء الكائن النهائي للعرض
  spotlight_db[player_key] = {
      "player_id": pid,
      "team": team,
      "role": role,
      "profile_stats": {
          "total_passes": stats["total_passes"],
          "pass_accuracy_pct": stats["pass_accuracy_pct"],
          "shots_total": stats["total_shots"],
          "shots_on_target": stats["shots_on_target"],
          "goals": stats["goals"],
          "defenders_packed": stats["total_defenders_packed"],
          "total_saves": stats["total_saves"],
          "total_defensive_actions": stats["total_defensive_actions"],
      },
      "moments_categories": {
          "key_actions": filtered_keys,
          "best_positioning": filtered_pos,
          "defensive_lapses": filtered_def,
      },
  }

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
  json.dump(spotlight_db, f, ensure_ascii=False, indent=2)

print("=" * 60)
print(f"🎉 تم بنجاح تصدير الملف النهائي: '{OUTPUT_FILE}'")
print(f"📊 إجمالي اللاعبين: {len(spotlight_db)} لاعب.")
print("✅ الإحصائيات شاملة في الكارت، واللقطات مفلترة لأقوى اللحظات فقط.")

⏳ جاري قراءة الملف الشامل: players_unfiltered_raw_profiles.json
🎉 تم بنجاح تصدير الملف النهائي: 'players_spotlight_profiles.json'
📊 إجمالي اللاعبين: 28 لاعب.
✅ الإحصائيات شاملة في الكارت، واللقطات مفلترة لأقوى اللحظات فقط.


=======================================================================================================================================================================================================================

In [1]:
import os
os.environ["GROQ_API_KEY"] = "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"  

    
import os
import json
import time
import pandas as pd
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
SELECTED_MODEL = "qwen/qwen3.8-27b"

In [ ]:
# ==============================================================================
# توليد شريط البث التكتيكي عبر Groq API مع حفظ تلقائي وحماية الـ Rate Limits
# المخرجات: live_broadcast_narratives.json
# ==============================================================================
import os
import json
import time
import pandas as pd
from groq import Groq
# 1. إعداد الـ API والموديل
# API_KEY = os.environ.get("GROQ_API_KEY", "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")
API_KEY = os.environ.get("GROQ_API_KEY", "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")

client = Groq(api_key=API_KEY)
# SELECTED_MODEL = "qwen/qwen3.8-27b"
SELECTED_MODEL = "allam-2-7b"



EVENTS_FILE = "Sample_Game_1_RawEventsData.csv" if os.path.exists("Sample_Game_1_RawEventsData.csv") else "events.csv"
OUTPUT_JSON = "live_broadcast_narratives.json"

print(f"⏳ جاري قراءة ملف الأحداث: {EVENTS_FILE}")
df_events = pd.read_csv(EVENTS_FILE)
df_events.columns = [str(c).strip() for c in df_events.columns]

# تصفية الأحداث الحركية فقط (تمريرات، تسديدات، استخلاص، تصديات)
action_mask = df_events['Type'].astype(str).str.upper().str.contains("PASS|SHOT|BALL LOST|RECOVERY|TACKLE")
key_events = df_events[action_mask].copy().reset_index(drop=True)

# 2. تحميل الشغل السابق إن وجد (لتفادي الإعادة لو الكود وقف)
# ==============================================================================
# كود الاستئناف الدقيق من آخر نقطة توقف
# ==============================================================================
generated_narratives = {}
if os.path.exists(OUTPUT_JSON):
    try:
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            existing_data = json.load(f)
            # ضمان تحويل الـ index لرقم صحيح للمقارنة الدقيقة
            for item in existing_data:
                if 'event_idx' in item:
                    generated_narratives[int(item['event_idx'])] = item
            print(f"🔄 تم العثور على {len(generated_narratives)} لقطة سابقة. سيتم الاستئناف فوراً من اللقطة التالية...")
    except Exception as e:
        print(f"تحذير أثناء القراءة: {e}")

# 3. دالة الـ Prompt والـ LLM Call
SYSTEM_PROMPT = """You are an elite TV Football Tactical Analyst and Broadcast Producer.
Your job is to generate a concise, broadcast-ready TV Lower-Third banner headline explaining the tactical significance of a match moment (Why it matters).
You MUST respond with a raw valid JSON object ONLY, strictly matching this structure:
{
  "ticker_ar": "تحليل تكتيكي سريع ومركز في سطر واحد لا يتجاوز 12 كلمة",
  "ticker_en": "One-line tactical headline under 12 words"
}
Do not include markdown fences, backticks, or extra explanation. Only JSON."""

total_events = len(key_events)
print(f"🚀 بدء معالجة {total_events} لقطة عبر موديل {SELECTED_MODEL}...")

# حفظ دوري كل 5 أحداث
SAVE_INTERVAL = 5

for i, row in key_events.iterrows():
    if i in generated_narratives:
        continue  # اللقطة دي اتعملت قبل كده، عديها
    
    start_s = float(row.get('Start Time [s]', 0))
    end_s = float(row.get('End Time [s]', start_s + 3.0))
    time_str = f"{int(start_s // 60):02d}:{int(start_s % 60):02d}"
    
    team = str(row.get('Team', 'Team'))
    player = str(row.get('From', 'Player')).replace('Player', 'Player ')
    ev_type = str(row.get('Type', 'Play')).upper()
    subtype = str(row.get('Subtype', '')).upper()
    
    # صياغة السياق التكتيكي
    user_context = f"""Match Event:
- Time: {time_str}
- Team: {team}
- Player: {player}
- Action: {ev_type}
- Details: {subtype}

Summarize why this moment matters tactically for a live TV Lower-Third banner."""

    success = False
    retries = 3
    
    while not success and retries > 0:
        try:
            response = client.chat.completions.create(
                model=SELECTED_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_context}
                ],
                temperature=0.3,
                max_tokens=120
            )
            
            raw_text = response.choices[0].message.content.strip()
            # تنظيف أي زوائد
            if raw_text.startswith("```"):
                raw_text = raw_text.split("```")[1]
                if raw_text.startswith("json"):
                    raw_text = raw_text[4:].strip()
            
            parsed = json.loads(raw_text)
            
            generated_narratives[i] = {
                "event_idx": i,
                "start_s": round(start_s, 1),
                "end_s": round(end_s, 1),
                "time_str": time_str,
                "team": team,
                "player": player,
                "action": ev_type,
                "subtype": subtype,
                "ticker_ar": parsed.get("ticker_ar", f"بناء لعب وتمركز تكتيكي من {player}"),
                "ticker_en": parsed.get("ticker_en", f"Tactical build-up by {player}")
            }
            
            print(f"[{i+1}/{total_events}] ({time_str}) {player} -> {parsed.get('ticker_ar', '')}")
            success = True
            
        except Exception as e:
            err_str = str(e).lower()
            retries -= 1
            if "rate_limit" in err_str or "429" in err_str:
                print(f"⚠️ تجاوز معدل الطلبات (Rate limit). انتظام 15 ثانية...")
                time.sleep(15)
            else:
                print(f"⚠️ خطأ: {e}. محاولة متبقية: {retries}")
                time.sleep(3)

    # حفظ تلقائي مؤقت
    if len(generated_narratives) % SAVE_INTERVAL == 0:
        out_list = sorted(list(generated_narratives.values()), key=lambda x: x['start_s'])
        with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
            json.dump(out_list, f, ensure_ascii=False, indent=2)

    # فاصل زمني آمن لتجنب حظر الـ Rate Limits
    time.sleep(1.5)

# 4. الحفظ النهائي
out_list = sorted(list(generated_narratives.values()), key=lambda x: x['start_s'])
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(out_list, f, ensure_ascii=False, indent=2)

print("=" * 60)
print(f"🎉 تم الانتهاء بنجاح وحفظ كامل نصوص البث في: '{OUTPUT_JSON}'")
print(f"📊 إجمالي اللقطات المغطاة: {len(out_list)} لقطة.")

⏳ جاري قراءة ملف الأحداث: Sample_Game_1_RawEventsData.csv
🔄 تم العثور على 855 لقطة سابقة. سيتم الاستئناف فوراً من اللقطة التالية...
🚀 بدء معالجة 1358 لقطة عبر موديل allam-2-7b...
[856/1358] (59:59) Player 5 -> لاعب 5 يُمرر كرة حاسمة في الوقت القاتل لفريق Home
[857/1358] (60:00) Player 10 -> لاعب 10 يسدد هدفًا محققًا لفريق المنزل في الدقيقة 60
[858/1358] (61:08) Player 19 -> في الدقيقة 61: تمريرة حاسمة من اللاعب 19 تُظهر مهارة الفريق الزائر التكتيكية
[859/1358] (61:12) Player 21 -> في الدقيقة 61، تمريرة حاسمة من اللاعب 21 للفريق الضيف تحمل دلالة تكتيكية
[860/1358] (61:15) Player 17 -> في الدقيقة 61:15، تمريرة حاسمة من لاعب الفريق الزائر رقم 17 تغير مجرى المباراة تكتيكيًا
[861/1358] (61:16) Player 15 -> في الدقيقة 61: خطأ من لاعب الفريق الزائر رقم 15 يؤدي لاعتراض الكرة
[862/1358] (61:18) Player 5 -> في الدقيقة 61:18.. الفريق المضيف يستعيد الكرة عبر اللاعب 5.. أهمية تكتيكية
[863/1358] (61:19) Player 5 -> في الدقيقة 61 والنتيجة 1-1، تمريرة حاسمة من اللاعب 5 تعزز هجوم الفريق المضيف
[864/135

match_recap

In [4]:
import json
import os
import numpy as np
import pandas as pd

# ==============================================================
# 1. إعداد الثوابت ومسارات الملفات
# ==============================================================
PITCH_L, PITCH_W = 105.0, 68.0
FPS = 25

HOME_TRACKING = "Sample_Game_1_RawTrackingData_Home_Team.csv"
AWAY_TRACKING = "Sample_Game_1_RawTrackingData_Away_Team.csv"
EVENTS_FILE = (
    "Sample_Game_1_RawEventsData.csv"
    if os.path.exists("Sample_Game_1_RawEventsData.csv")
    else "events.csv"
)
NARRATIVES_FILE = "live_broadcast_narratives.json"
OUTPUT_FILE = "match_recap_data.json"


# ==============================================================
# 2. دوال مساعدة لقراءة الداتا وتطبيع الأبعاد
# ==============================================================
def read_tracking(csv_path, team_prefix):
  with open(csv_path, "r", encoding="utf-8") as f:
    _ = f.readline()
    row1 = f.readline().strip().split(",")
    _ = f.readline()

  col_names = ["Period", "Frame", "Time [s]"]
  i = 3
  while i < len(row1):
    if i >= len(row1) - 2:
      col_names.extend(["Ball_x", "Ball_y"])
      break
    num = str(row1[i]).strip() or str(row1[i - 1]).strip()
    col_names.append(f"{team_prefix}_Player{num}_x")
    col_names.append(f"{team_prefix}_Player{num}_y")
    i += 2

  df = pd.read_csv(
      csv_path, skiprows=3, header=None, names=col_names, low_memory=False
  )
  for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

  x_cols = [c for c in df.columns if c.endswith("_x")]
  y_cols = [c for c in df.columns if c.endswith("_y")]
  if x_cols and np.nanmax(df[x_cols].to_numpy(dtype=float)) <= 1.5:
    for c in x_cols:
      df[c] = df[c] * PITCH_L
    for c in y_cols:
      df[c] = (1.0 - df[c]) * PITCH_W
  return df


print("1/4 Loading Tracking and Event data...")
df_home = read_tracking(HOME_TRACKING, "Home")
df_away = read_tracking(AWAY_TRACKING, "Away")
df_events = pd.read_csv(EVENTS_FILE) if os.path.exists(EVENTS_FILE) else None

narratives = []
if os.path.exists(NARRATIVES_FILE):
  with open(NARRATIVES_FILE, "r", encoding="utf-8") as f:
    narratives = json.load(f)

# ==============================================================
# 3. حساب الأهداف والنتيجة النهائية
# ==============================================================
print("2/4 Parsing Goals & Scoreboard...")
goals = []
home_goals = 0
away_goals = 0

if df_events is not None and {"Type", "Subtype", "Team"}.issubset(
    df_events.columns
):
  g_df = df_events[
      df_events["Type"].astype(str).str.upper().str.contains("SHOT")
      & df_events["Subtype"].astype(str).str.upper().str.contains("GOAL")
  ]
  for _, r in g_df.iterrows():
    team = str(r["Team"])
    if team == "Home":
      home_goals += 1
    elif team == "Away":
      away_goals += 1

    time_s = (
        float(r["End Time [s]"])
        if pd.notna(r.get("End Time [s]"))
        else (float(r["Start Frame"]) / FPS)
    )
    minute = int(time_s // 60)
    second = int(time_s % 60)
    player = str(r.get("From", "Unknown"))
    goals.append({
        "team": team,
        "player": player,
        "minute": minute,
        "second": second,
        "clock": f"{minute:02d}:{second:02d}",
        "time_s": round(time_s, 2),
    })

match_duration_s = float(df_home["Time [s]"].max() - df_home["Time [s]"].min())

# ==============================================================
# 4. حساب المسافات المقطوعة وسرعات اللاعبين (من التراكينج)
# ==============================================================
print("3/4 Calculating Distances & Physical Intensity...")


def compute_player_distances(df, prefix):
  x_cols = [
      c for c in df.columns if c.startswith(prefix) and c.endswith("_x")
  ]
  player_stats = {}
  total_team_dist = 0.0

  # عينة كل 5 فريمات لتسريع الحساب بدقة عالية
  sampled = df.iloc[::5]
  dt = 5.0 / FPS

  for xc in x_cols:
    yc = xc[:-2] + "_y"
    p_name = xc[len(prefix) + 1 : -2]
    xs = sampled[xc].to_numpy(dtype=float)
    ys = sampled[yc].to_numpy(dtype=float)

    dx = np.diff(xs)
    dy = np.diff(ys)
    step_dists = np.sqrt(dx**2 + dy**2)

    # فلترة القفزات غير الواقعية (البيانات المفقودة)
    valid_steps = step_dists[
        (step_dists < 25.0) & (~np.isnan(step_dists))
    ]  # أقل من سرعة 50 م/ث
    p_dist = float(np.sum(valid_steps))
    player_stats[p_name] = round(p_dist, 1)
    total_team_dist += p_dist

  return player_stats, round(total_team_dist / 1000.0, 2)  # بالكيلومتر


home_player_dist, home_total_km = compute_player_distances(df_home, "Home")
away_player_dist, away_total_km = compute_player_distances(df_away, "Away")

# ==============================================================
# 5. حساب السيطرة الميدانية وأثلاث الملعب (Territory & Thirds)
# ==============================================================
# استخدام إحداثيات الكرة عبر الـ 90 دقيقة
bx = df_home["Ball_x"].dropna().to_numpy(dtype=float)
# تقليم القيم خارج حدود الملعب
bx = bx[(bx >= 0.0) & (bx <= PITCH_L)]

total_ball_frames = len(bx)
if total_ball_frames > 0:
  def_third = float(np.sum(bx < 35.0) / total_ball_frames * 100.0)
  mid_third = float(
      np.sum((bx >= 35.0) & (bx <= 70.0)) / total_ball_frames * 100.0
  )
  att_third = float(np.sum(bx > 70.0) / total_ball_frames * 100.0)
else:
  def_third, mid_third, att_third = 33.3, 33.4, 33.3

# ==============================================================
# 6. المقاييس التكتيكية والرادار (من الـ Events والـ Narratives)
# ==============================================================
print("4/4 Aggregating Tactical Metrics & Leaderboards...")

# استخراج مقاييس التمرير والتسديد
pass_events = (
    df_events[df_events["Type"].astype(str).str.upper().str.contains("PASS")]
    if df_events is not None
    else pd.DataFrame()
)
shot_events = (
    df_events[df_events["Type"].astype(str).str.upper().str.contains("SHOT")]
    if df_events is not None
    else pd.DataFrame()
)
recovery_events = (
    df_events[
        df_events["Type"].astype(str).str.upper() == "RECOVERY"
    ]
    if df_events is not None
    else pd.DataFrame()
)
home_passes = len(pass_events[pass_events["Team"] == "Home"])
away_passes = len(pass_events[pass_events["Team"] == "Away"])

home_shots = len(shot_events[shot_events["Team"] == "Home"])
away_shots = len(shot_events[shot_events["Team"] == "Away"])

home_recoveries = len(recovery_events[recovery_events["Team"] == "Home"])
away_recoveries = len(recovery_events[recovery_events["Team"] == "Away"])
print(f"Recoveries -> Home: {home_recoveries} | Away: {away_recoveries}")

# استخراج الـ Packing والـ Line Breaks من ملف الـ Narratives
player_impact = {}
total_line_breaks = {"Home": 0, "Away": 0}
total_packing_points = {"Home": 0, "Away": 0}

for n in narratives:
  team = n.get("team", "")
  player = n.get("player", "")
  action = n.get("action", "").upper()

  # حساب الأثر التكتيكي لكل لاعب
  if player:
    if player not in player_impact:
      player_impact[player] = {
          "player": player,
          "team": team,
          "line_breaks": 0,
          "packing_score": 0,
          "involvements": 0,
      }
    player_impact[player]["involvements"] += 1

    # استخلاص كسر الخطوط
    text_check = (
        n.get("ticker_en", "").lower() + " " + n.get("ticker_ar", "")
    )
    if "line break" in text_check or "كسر خط" in text_check:
      if team in total_line_breaks:
        total_line_breaks[team] += 1
      player_impact[player]["line_breaks"] += 1
      player_impact[player]["packing_score"] += 3  # تقدير 3 مدافعين تم تجاوزهم

    elif "progressive" in text_check or "تقدم" in text_check:
      player_impact[player]["packing_score"] += 2

# ترتيب أفضل 5 لاعبين تأثيراً
top_players = sorted(
    player_impact.values(),
    key=lambda x: (x["packing_score"], x["line_breaks"], x["involvements"]),
    reverse=True,
)[:5]

# دمج المسافة المقطوعة للاعبين الأفضل إن وُجدت
for p in top_players:
  num = "".join(filter(str.isdigit, p["player"]))
  dist_dict = home_player_dist if p["team"] == "Home" else away_player_dist
  p["distance_m"] = dist_dict.get(
      f"Player{num}", dist_dict.get(num, 0.0)
  )  # بالمتر

# Field Tilt (نسبة التمرير في الثلث الأخير)
home_final_third_passes = int(home_passes * (att_third / 100.0))
away_final_third_passes = int(away_passes * (def_third / 100.0))
tot_ft = max(1, home_final_third_passes + away_final_third_passes)
home_field_tilt = round((home_final_third_passes / tot_ft) * 100.0, 1)
away_field_tilt = round(100.0 - home_field_tilt, 1)

# ==============================================================
# 7. تجميع وتصدير ملف الـ JSON الشامل
# ==============================================================
recap_data = {
    "scoreboard": {
        "home_score": home_goals,
        "away_score": away_goals,
        "match_duration_min": round(match_duration_s / 60.0, 1),
        "goals": goals,
    },
    "kpis": {
      "total_distance_km": round(home_total_km + away_total_km, 1),
      "home_distance_km": home_total_km,
      "away_distance_km": away_total_km,
      "total_line_breaks": (
          total_line_breaks["Home"] + total_line_breaks["Away"]
      ),
      "total_packing_events": sum(
          p["packing_score"] for p in player_impact.values()
      ),
      "home_field_tilt": home_field_tilt,
      "away_field_tilt": away_field_tilt,
  },
  "territory_thirds": {
      "defensive_third_pct": round(def_third, 1),
      "middle_third_pct": round(mid_third, 1),
      "attacking_third_pct": round(att_third, 1),
  },
  "radar_comparison": {
      "metrics": [
          "Passes",
          "Line Breaks",
          "Shots",
          "Recoveries",
          "Final 3rd Tilt",
      ],
      "home": [
          home_passes,
          total_line_breaks["Home"],
          home_shots,
          home_recoveries,
          int(home_field_tilt),
      ],
      "away": [
          away_passes,
          total_line_breaks["Away"],
          away_shots,
          away_recoveries,
          int(away_field_tilt),
      ],
  },
  "top_impact_players": top_players,
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
  json.dump(recap_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ SUCCESS! Recap data saved to: {OUTPUT_FILE}")
print(
    f"Score: HOME {home_goals} - {away_goals} AWAY | Goals detected:"
    f" {len(goals)}"
)
print(f"Top Impact Player: {top_players[0]['player']} ({top_players[0]['team']})")

1/4 Loading Tracking and Event data...
2/4 Parsing Goals & Scoreboard...
3/4 Calculating Distances & Physical Intensity...
4/4 Aggregating Tactical Metrics & Leaderboards...
Recoveries -> Home: 135 | Away: 143

✅ SUCCESS! Recap data saved to: match_recap_data.json
Score: HOME 3 - 0 AWAY | Goals detected: 3
Top Impact Player: Player 3 (Home)


In [5]:
import json
import os
from groq import Groq

# 1. إعداد الـ Client والموديل المعتمد عندك
API_KEY = os.environ.get("GROQ_API_KEY", "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")
client = Groq(api_key=API_KEY)
SELECTED_MODEL = "allam-2-7b"

JSON_PATH = "match_recap_data.json"
with open(JSON_PATH, "r", encoding="utf-8") as f:
  recap_data = json.load(f)

# 2. صياغة الـ Prompt التكتيكي المركز
prompt = f"""
You are a Senior Football Tactical Analyst and Executive Match Producer.
Analyze the following post-match aggregated statistics and write an elite, professional tactical summary:

MATCH DATA:
{json.dumps(recap_data, indent=2)}

REQUIREMENTS:
Return ONLY a valid JSON object with the following exact keys:
- "summary_en": Executive tactical summary in English (3-4 sentences explaining territorial dominance, finishing efficiency, and progression).
- "summary_ar": Same tactical summary in fluent, authentic Arabic tactical terminology.
- "turning_point_en": The decisive moment of the match in English.
- "turning_point_ar": The decisive moment in Arabic.
"""

print(f"Sending request to Groq using model: {SELECTED_MODEL}...")

response = client.chat.completions.create(
    model=SELECTED_MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are an expert football analytics AI. You always respond"
                " strictly in valid JSON format without markdown wrapping or"
                " explanations."
            ),
        },
        {"role": "user", "content": prompt},
    ],
    response_format={"type": "json_object"},
    temperature=0.2,
)

# 3. استخراج الـ JSON وحفظه داخل الملف
narrative_output = json.loads(response.choices[0].message.content)
recap_data["narrative"] = narrative_output

with open(JSON_PATH, "w", encoding="utf-8") as f:
  json.dump(recap_data, f, ensure_ascii=False, indent=2)

print("\n✅ Successfully integrated Groq AI Narrative into match_recap_data.json!")
print(f"Summary (EN): {narrative_output.get('summary_en')}")
print(f"Summary (AR): {narrative_output.get('summary_ar')}")

Sending request to Groq using model: allam-2-7b...

✅ Successfully integrated Groq AI Narrative into match_recap_data.json!
Summary (EN): {'Territorial dominance': 'The home team dominated the match in terms of territory, holding 67.4% of possession throughout the match.', 'Finishing efficiency': "The home team's finishing efficiency was impressive, converting 3 out of 6 shots on target.", 'Progression': "The home team's progression was effective, creating 11 line breaks and making 27 tackles."}
Summary (AR): {'تفوق ميداني': 'فريق المضيف سيطر على المباراة من حيث السيطرة الميدانية بنسبة 67.4%.', 'فعالية التسديد': 'فعالية تسديد الفريق المضيف كانت مميزة، حيث سجلوا 3 أهداف من أصل 6 تسديدات على المرمى.', 'تحقيق التقدم': 'تحقيق التقدم كان فعّالاً، حيث قام الفريق المضيف بخلق 11 فرصة تجاوز الخصم و18 تدخلات'}
